# Cellpose-Based Segmentation for Extracting Morphological Features from Drug-Treated Cancer Cell Images

High-content imaging (high-throughput microscopy) is widely used to study how biological samples respond to chemical exposure. For example, some drug compounds may change morphological characteristics in cells by affecting cytoskeleton, DNA replication, or protein regulation. These changes create a measurable “phenotypic fingerprint”: cells may change their shape, size, nuclear morphology, actin organisation, or microtubule structure depending on the compound’s mechanism of action.

However, manual evaluation of microscopy images misses many details required for comparison. Therefore, computational phenotypic profiling is used. It converts microscopy images into numerical features that can capture even subtle cellular changes and enable classification of drug mechanisms of action [(Ljosa et al., 2013)](https://journals.sagepub.com/doi/10.1177/1087057113503553).

This project uses the [BBBC021](https://bbbc.broadinstitute.org/BBBC021) image set from the Broad Bioimage Benchmark Collection. The project focuses on an interpretable subset of cells treated with 5 compounds.

The aim of the project is to develop a pipeline for cell segmentation, extracting morphometric features, and comparing cellular phenotypes induced by different compounds. This strategy was introduced in previous studies by [Caie et al. (2010)](https://aacrjournals.org/mct/article/9/6/1913/93798/High-Content-Phenotypic-Profiling-of-Drug-Response) and [Ljosa et al. (2013)](https://journals.sagepub.com/doi/10.1177/1087057113503553).

# Libraries and set-up

In [ ]:
!pip -q install cellpose==3.1.1.2 stardist tensorflow

In [ ]:
# for DT visual
!apt-get -qq install graphviz
!pip -q install dtreeviz

In [ ]:
# Standard
import io
import json
import os
import random
import shutil
import warnings
import numpy as np
import pandas as pd
import zipfile
from itertools import product
from pathlib import Path
import requests
from tqdm.auto import tqdm

# Plotting
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
from matplotlib.colors import ListedColormap
import dtreeviz
from IPython.display import SVG, display

# Image processing
import scipy.ndimage as ndi
import tifffile
from skimage import exposure, filters, measure, morphology, segmentation
from skimage.color import label2rgb
from skimage.feature import peak_local_max
from skimage.segmentation import find_boundaries

# Statistics and dimensionality reduction
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Classical ML
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Deep learning and segmentation
import torch
from cellpose import models as cp_models
from cellpose import utils as cp_utils
from csbdeep.utils import normalize
from stardist.models import StarDist2D

In [ ]:
# Reproducibility
SEED = 42

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)

# GPU Check, Cellpose on CPU is 30 times slower
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA is available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}  ({props.total_memory / 1e9:.1f} GB)")
print(f"DEVICE = {DEVICE}")

In [ ]:
# Catalogs
BASE_DIR = Path("/content/bbbc021")
DATA_DIR = BASE_DIR
META_DIR = DATA_DIR / "metadata"
TIFF_DIR = DATA_DIR / "tiffs"
CACHE_DIR = DATA_DIR / "cache"
MASKS_DIR = CACHE_DIR / "masks"
FIGURES_DIR = DATA_DIR / "figures"

for d in (DATA_DIR, META_DIR, TIFF_DIR, CACHE_DIR, MASKS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Folders created:")
for d in (DATA_DIR, META_DIR, TIFF_DIR, CACHE_DIR, MASKS_DIR, FIGURES_DIR):
    print(f"  {d}")

# Meet Dataset and choose plates

The project uses the image set [BBBC021v1](https://bbbc.broadinstitute.org/bbbc/BBBC021) ([Caie et al., 2010](http://dx.doi.org/10.1158/1535-7163.MCT-09-1148)), available from the Broad Bioimage Benchmark Collection ([Ljosa et al., 2012](http://dx.doi.org/10.1038/nmeth.2083)). Specifically, microscopy images of MCF-7 breast cancer cells treated for 24 h with a collection of 113 small molecules at eight concentrations. The cells were fixed, labeled for DNA, F-actin, and Β-tubulin, and imaged by fluorescent microscopy ([Caie et al. 2010](http://dx.doi.org/10.1158/1535-7163.MCT-09-1148)). Metadata contains linking images to compounds, plates, wells, and concentrations, mechanism-of-action for selected compound-concentration pairs.

**Channels description:**

| **Channel** | **Shows** | **Colour** |
|---|---|---|
| **DAPI** | **Nucleus** | **Blue** |
| **β-tubulin** | **Microtubules** | **Green** |
| **Actin** | **Cytoskeleton** | **Red** |

Each field of view is therefore acquired in 3 channels: the signal from nuclei separately, the signal from microtubules separately, and the signal from actin separately. Each channel is stored as a grayscale TIFF. Several cellular structures are included because a single structure does not describe the entire morphology of a cell. Nuclei help count cells and serve as markers for segmentation. Microtubules provide information about cytoskeletal organisation. The actin cytoskeleton provides information about cell boundaries and cell shape. In other words, DAPI helps show where the cells are, while tubulin and actin help reveal how drug treatment changes cellular structure.

DAPI (4',6-diamidino-2-phenylindole) is a fluorescent dye. It binds to DNA, so in the image it shows the cell nucleus. β-tubulin is a cellular protein from which microtubules are built; microtubules are elements of the cytoskeleton, the internal “scaffold” of the cell. β-tubulin is detected using an antibody: an antibody with a fluorescent label binds to β-tubulin, and the microscope captures this signal. Actin is also a cellular protein and also a component of the cytoskeleton. The actin cytoskeleton is especially important for cell shape, cell edges, movement, and attachment to a surface. Microtubules, by contrast, are more involved in intracellular transport and cell division. Historically, actin was studied as a protein involved in muscle contraction. However, it is not found only in muscles, but in almost all eukaryotic cells. In fluorescence images, actin is stained with phalloidin carrying a fluorescent label.

In [ ]:
BBBC021_BASE_URL = "https://data.broadinstitute.org/bbbc/BBBC021"

META_FILES = [
    "BBBC021_v1_image.csv",
    "BBBC021_v1_moa.csv",
]

for name in META_FILES:
    url = f"{BBBC021_BASE_URL}/{name}"
    dest = META_DIR / name

    response = requests.get(url, timeout=60)
    dest.write_bytes(response.content)

    print(f"Downloaded: {dest.name} ({dest.stat().st_size / 1024:.1f} KB)")

The metadata are stored in two CSV files:

BBBC021_v1_image.csv — an index of all images: for each field of view, it contains the filenames of the three channels, the plate, well, site, concentration, and compound name.

BBBC021_v1_moa.csv — a lookup table mapping each compound to its mechanism of action (MOA).

In [ ]:
images_meta = pd.read_csv(META_DIR / "BBBC021_v1_image.csv")
moa_meta = pd.read_csv(META_DIR / "BBBC021_v1_moa.csv")

print("images_meta:", images_meta.shape)
display(images_meta.head())

print("Columns:")
list(images_meta.columns)

In the `images_meta` table (from `BBBC021_v1_image.csv`), each row represents one field of view. There are 13,200 fields in total. For each field, the table contains information about which three TIFF files correspond to the same location on the plate, which compound the cells were treated with, the concentration, and the well. We use this table to get the image paths and select the required ZIP archives.

In total, the dataset contains 113 compounds × 8 concentrations × 3 replicates.
All images were acquired in 55 96-well plates, which are laboratory plates with wells where cell cultures are grown. One plate corresponds to one ZIP archive (~700 MB). Each plate has 96 wells; each well contains either the control condition (DMSO) or one treatment with one compound at one concentration. A site / field of view is one photographed area within a well, usually 4 sites per well. The three TIFF files are the three channels of the same site.

plate

  └── well

        └── site / field of view

              ├── DAPI TIFF

              ├── Tubulin TIFF

              └── Actin TIFF

Each image is a microscopic field of view containing several dozen MCF-7 cells.

In [ ]:
print(f"Unique compounds: {images_meta['Image_Metadata_Compound'].nunique()}")
print(f"Unique plates: {images_meta['Image_Metadata_Plate_DAPI'].nunique()}")

print(f"Unique MoA: {moa_meta['moa'].nunique()}")
print("Shape of moa_meta:", moa_meta.shape)

display(moa_meta.head(10))

In [ ]:
compounds_by_moa = (
    moa_meta
    .groupby("moa")["compound"]
    .apply(lambda x: sorted(x.dropna().unique()))
    .reset_index(name="compounds")
)

display(compounds_by_moa)

From the table, it can be seen that all compounds in the dataset can be divided into 12 mechanisms of action, not counting one DMSO control. There are actin disruptors, Aurora kinase inhibitors, DNA damage mechanisms, etc.

For this study, I would like to choose compounds that are as comparable as possible.

Firstly, compounds from different MoA groups should be selected. This way, we will probably obtain the most morphologically distinct cell phenotypes. This is not guaranteed, but it is likely.

The aim of the project is not to conduct an exhaustive analysis, but to test whether, in principle, it is possible to distinguish some phenotypes of cells treated with different compounds using machine learning.

Secondly, the compounds have different concentrations, and even different concentration ranges, which also leads us to the assumption that we need to choose compounds with a similar concentration profile.

For example, the selected compounds should have at least three concentrations each, so that we can first choose a middle concentration for comparing the compounds, and then possibly expand the project to analyse all three concentrations in order to understand whether cell morphology differs depending on concentration.

In [ ]:
compound_concentrations = (
    moa_meta
    .groupby(["moa", "compound"], as_index=False)
    .agg(
        concentrations=("concentration", lambda x: sorted(x.dropna().unique()))
    )
    .sort_values(["moa", "compound"])
)

display(compound_concentrations)

It is important to understand that this table does not contain all possible concentrations. We know that the dataset includes 8 concentrations for each compound. However, the authors did not leave all of them in the metadata table. Most likely, this is because the dataset authors selected the most meaningful concentrations. Therefore, it is more logical to choose from these.

In order not to make the dataset too large while testing the hypothesis, we will choose 4 compounds from different MoA groups among those that have 3 concentrations of a close order of magnitude, plus DMSO as the control.

I would note that we will not compare numerically equal concentrations of different compounds, because this would be incorrect. A concentration of 1.0 for Taxol and 1.0 for Etoposide are not necessarily biologically comparable in terms of effect strength. One compound may already have an effect at the minimum concentration, while another may require a 10 times higher concentration to produce some effect.

Therefore, the compromise chosen for the project is that each selected compound will have a low, medium, and high dose within that compound itself.

In [ ]:
candidates_3 = compound_concentrations[
    compound_concentrations["concentrations"].apply(len) == 3
].sort_values(["moa", "compound"])

display(candidates_3)

Having a list of compounds with 3 available concentrations, we narrow the choice down to 7 compounds. Then we select compounds from different MoA groups with concentrations of a similar order of magnitude. For now, we will stop at:

- control                         DMSO            [0.0]
- Aurora kinase inhibitor         AZ258           [0.1, 0.3, 1.0]
- DNA replication                  camptothecin    [0.003, 0.01, 0.03]
- Microtubule stabilizer          docetaxel       [0.03, 0.1, 0.3]
- Microtubule stabilizer          epothilone B    [0.1, 0.3, 1.0]
- Protein synthesis                emetine         [0.1, 0.3, 1.0]

Approximate expectations for morphology:

DMSO is a solvent, used as a control to avoid confusing the effect of the solvent with the effect of the drug; it is not expected to do anything and provides normal cells for comparison.

Camptothecin is associated with problems during DNA replication through topoisomerase I, which makes one temporary cut in DNA. Expected effects include changes in DAPI, cell-cycle arrest/death of some cells, and a possible decrease in cell density (https://www.cancer.gov/publications/dictionaries/cancer-drug/def/camptothecin).

Emetine is a protein synthesis inhibitor; cells are expected to be unable to properly renew their components, become shrunken and less spread, with altered actin/cytoplasm morphology. The cells often look more difficult to segment: small, compressed, dense, and sometimes clustered (https://meshb.nlm.nih.gov/record/ui?ui=D004640).

AZ258 is an Aurora kinase inhibitor, disrupting the process of cell division regulated by Aurora kinase; expected effects include disruption of the mitotic spindle, chromosome segregation errors, cell-cycle arrest, apoptosis, cells becoming stuck in mitosis, rounded morphology, and abnormal nuclei (https://www.cancer.gov/publications/dictionaries/cancer-drug/def/aurora-a-kinase-inhibitor-mk5108).

Docetaxel is a microtubule stabilizer from the taxane group; it binds to tubulin, stabilizes microtubules, and prevents their normal disassembly. Expected effects include cell-cycle arrest in G2/M, inhibition of mitosis, cell death in some cells, rounded mitotic cells, reduced spreading, and changes in the tubulin channel because the microtubule network is directly affected (https://www.cancer.gov/publications/dictionaries/cancer-drug/def/docetaxel).

Epothilone B is also a microtubule stabilizer; similarly to taxanes, it promotes microtubule polymerization and stabilizes microtubules against depolymerization. Expected effects include mitotic arrest, reduced cell proliferation, possible apoptosis, rounded or arrested cells, and strong changes in the tubulin structure (https://www.cancer.gov/publications/dictionaries/cancer-drug/def/patupilone).

Each plate is a separate ZIP file of about 700 MB. Downloading all 55 plates would be about 40 GB, which is not necessary for this analysis. We want to download the minimum number of ZIP archives while covering as many of the required compounds as possible.

Therefore, we need to filter and download plates that contain the largest possible number of compounds selected for analysis at the same time.

In [ ]:
COMPOUND_DISPLAY = {
    "dmso": "DMSO",
    "mevinolin/lovastatin": "Lovastatin",
    "az-u": "AZ-U",
    "etoposide": "Etoposide",
    "az258": "AZ258",
}

COMPOUND_ORDER = list(COMPOUND_DISPLAY.values())

# normalization of names
def normalize_compound(name: object) -> str:
    name_lower = str(name).strip().lower()
    return COMPOUND_DISPLAY.get(name_lower, "other")

images_meta["compound_norm"] = images_meta["Image_Metadata_Compound"].apply(normalize_compound)

sub_meta = (
    images_meta[
        images_meta["compound_norm"].isin(COMPOUND_ORDER)
    ]
    .reset_index(drop=True)
)

print(f"Fields with needed compounds: {len(sub_meta)}")
print()
print("Compound distribution:")
print(sub_meta["compound_norm"].value_counts().reindex(COMPOUND_ORDER, fill_value=0))

In [ ]:
# Which plates cover most
plate_coverage = (
    sub_meta.groupby("Image_Metadata_Plate_DAPI")["compound_norm"]
    .nunique()
    .sort_values(ascending=False)
)
print("Top-10 plates with variety of compounds:")
print(plate_coverage.head(10))

In [ ]:
plate_compounds = (
    sub_meta
    .groupby("Image_Metadata_Plate_DAPI")["compound_norm"]
    .apply(lambda x: sorted(x.unique()))
    .sort_values(key=lambda s: s.apply(len), ascending=False)
)

display(plate_compounds.head(12))

In [ ]:
SELECTED_PLATES = [
    "Week9_39221",  # DMSO, Lovastatin
    "Week4_27821",  # AZ258, DMSO
    "Week10_40115",  # AZ-U, DMSO
    "Week3_25441",  # Etoposide, DMSO
]

plates_meta = sub_meta[
    sub_meta["Image_Metadata_Plate_DAPI"].isin(SELECTED_PLATES)
].copy()

# create a df with required plates and check if they were selected
print("Plates chosen:")
for plate in SELECTED_PLATES:
    compounds = sorted(plates_meta.loc[
        plates_meta["Image_Metadata_Plate_DAPI"] == plate,
        "compound_norm"
    ].unique())
    print(plate, compounds)

In [ ]:
concentrations = (
    plates_meta
    .groupby("compound_norm")["Image_Metadata_Concentration"]
    .apply(lambda x: sorted(x.dropna().unique()))
    .reindex(COMPOUND_ORDER)
)

display(concentrations)

`moa_meta` is not the complete experiment index, but a ground truth table for mechanisms of action. It contains only those compound-concentration pairs that the dataset authors considered meaningful for MOA classification.

Some doses were not included in this MOA subset because they were inactive, overly toxic, or did not pass QC. Since this project is built around MOA and phenotypic comparison, it is better to take the concentrations from `moa_meta`. If we used all concentrations from `images_meta`, we could include doses that may have been inactive, toxic, or QC-problematic for MOA analysis.

Therefore, we filter the data using the three selected concentrations.

In [ ]:
TARGET_CONCENTRATIONS = {
    "DMSO": [0.0],
    "Lovastatin": [1.5, 5.0, 15.0],
    "AZ-U": [1.0, 3.0, 10.0],
    "Etoposide": [1.0, 3.0, 10.0],
    "AZ258": [0.1, 0.3, 1.0],
}

MIDDLE_CONCENTRATIONS = {
    "DMSO": 0.000,
    "Lovastatin": 5.0,
    "AZ-U": 3.000,
    "Etoposide": 3.000,
    "AZ258": 0.300,
}

target_conditions = pd.DataFrame([
    {"compound_norm": compound, "Image_Metadata_Concentration": concentration}
    for compound, concentrations in TARGET_CONCENTRATIONS.items()
    for concentration in concentrations
])

plates_meta = plates_meta.merge(
    target_conditions,
    on=["compound_norm", "Image_Metadata_Concentration"],
    how="inner",
)

We will also take the middle concentrations of the compounds for the main analysis later. We will compare phenotypes at one middle concentration so that the comparison between compounds is valid. This assumption is based on the idea that the dataset authors estimated the effective concentrations for the compounds in a roughly comparable way.

Nevertheless, we cannot confidently claim that all observed differences are due to the drug effect rather than plate or well effects. Hundreds of cells from the same plate or field of view cannot be treated as fully independent observations.

We will also take DMSO from all plates to check whether the controls from different plates are comparable (no batch/plate effect).

In [ ]:
print(f"Total fields in images_meta: {len(images_meta)}")
print(f"Fields after compound filter: {len(sub_meta)}")

In [ ]:
# DMSO from the same week-family as the drug
MAX_FIELDS_PER_CONDITION = 4 # all we have
MAX_DMSO_FIELDS_PER_PLATE = 1

dmso_meta = plates_meta[plates_meta["compound_norm"] == "DMSO"].copy()
drug_meta = plates_meta[plates_meta["compound_norm"] != "DMSO"].copy()

dmso_meta = (
    dmso_meta
    .sample(frac=1.0, random_state=SEED)
    .groupby(["Image_Metadata_Plate_DAPI"], group_keys=False)
    .head(MAX_DMSO_FIELDS_PER_PLATE)
)

drug_meta = (
    drug_meta
    .sample(frac=1.0, random_state=SEED)
    .groupby(["compound_norm", "Image_Metadata_Concentration"], group_keys=False)
    .head(MAX_FIELDS_PER_CONDITION)
)

plates_meta = (
    pd.concat([dmso_meta, drug_meta], ignore_index=True)
    .reset_index(drop=True)
)

print(f"Final fields of view: {len(plates_meta)}")

print()
print("DMSO across selected plates:")
display(
    plates_meta[plates_meta["compound_norm"] == "DMSO"]
    .groupby("Image_Metadata_Plate_DAPI")
    .size()
    .reset_index(name="n_fields")
    .sort_values("Image_Metadata_Plate_DAPI")
)

print()
print("Drugs by concentration:")
display(
    plates_meta[plates_meta["compound_norm"] != "DMSO"]
    .groupby(["compound_norm", "Image_Metadata_Concentration"])
    .size()
    .reset_index(name="n_fields")
    .sort_values(["compound_norm", "Image_Metadata_Concentration"])
)

To save disk space, we download each ZIP file to a temporary file, extract only the required TIFF files using the filenames from `plates_meta`, and delete the ZIP after extraction.

After this step, `data/tiffs/` will contain 52 fields of view × 3 channels = 156 TIFF files.

# Download plates and extract TIFF

In [ ]:
def plate_zip_url(plate: str) -> str:
    return f"{BBBC021_BASE_URL}/BBBC021_v1_images_{plate}.zip"

def list_needed_tiffs(meta_subset: pd.DataFrame) -> set[str]:
    cols = ["Image_FileName_DAPI", "Image_FileName_Tubulin", "Image_FileName_Actin"]
    return set(meta_subset[cols].values.ravel())

def download_large_file(url: str, dest: Path, chunk_size: int = 1024 * 1024) -> None:
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)

def extract_subset_from_zip(zip_path: Path, want_names: set[str], out_dir: Path) -> int:
    extracted = 0

    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            filename = Path(info.filename).name

            if filename in want_names:
                out_path = out_dir / filename

                with zf.open(info) as src, open(out_path, "wb") as dst:
                    shutil.copyfileobj(src, dst)

                extracted += 1

    return extracted

needed = list_needed_tiffs(plates_meta)

print(f"Will unzip {len(needed)} TIFF from {len(SELECTED_PLATES)} ZIP.")

In [ ]:
# Main loading loop
for plate in SELECTED_PLATES:
    plate_meta = plates_meta[
        plates_meta["Image_Metadata_Plate_DAPI"] == plate
    ]

    needed_plate = list_needed_tiffs(plate_meta)
    missing = [name for name in needed_plate if not (TIFF_DIR / name).exists()]

    if not missing:
        print(f"{plate}: all {len(needed_plate)} TIFF files are already extracted")
        continue

    zip_path = DATA_DIR / f"{plate}.zip"
    url = plate_zip_url(plate)

    print(f"{plate}: downloading ZIP, need to extract {len(missing)} TIFF files")
    download_large_file(url, zip_path)

    n_extracted = extract_subset_from_zip(zip_path, set(missing), TIFF_DIR)
    print(f"{plate}: extracted {n_extracted} TIFF files")

    zip_path.unlink()
    print(f"{plate}: ZIP removed")

In [ ]:
def field_files_exist(row: pd.Series) -> bool:
    """
    Сhecks if each field contains all three channels
    If all three are here, files_ok = True
    If at least one is missing, files_ok = False,
    problem rows are displayed

    """
    return all(
        (TIFF_DIR / row[col]).exists()
        for col in ["Image_FileName_DAPI", "Image_FileName_Tubulin", "Image_FileName_Actin"]
    )

plates_meta["files_ok"] = plates_meta.apply(field_files_exist, axis=1)

print(f"Fields with full set of channels: {plates_meta['files_ok'].sum()} / {len(plates_meta)}")

if not plates_meta["files_ok"].all():
    missing_rows = plates_meta.loc[
        ~plates_meta["files_ok"],
        [
            "compound_norm",
            "Image_Metadata_Concentration",
            "Image_Metadata_Plate_DAPI",
            "Image_FileName_DAPI",
            "Image_FileName_Tubulin",
            "Image_FileName_Actin",
        ],
    ]

    display(missing_rows)
    raise RuntimeError("Not all TIFFs extracted. Need to handle missing files")

# Image preprocessing

Each field of view consists of 3 separate TIFF files of the same type (`uint16`, 1024×1280).

Steps:

1. **Read each channel** using `tifffile.imread`.
2. **Convert to `float32` and normalize to [0, 1]** by clipping the darkest 1% and brightest 1% of pixels using percentiles.
3. **Build an RGB composite** for visualization: red = actin, green = tubulin, blue = DAPI.

Fluorescence images almost always contain outliers: individual extremely bright pixels, either artifacts or especially bright spots within cells. If we do not clip them, standard min-max normalization can make the rest of the image look gray. Therefore, percentile clipping at 1% and 99% is commonly used.

In [ ]:
def normalize_percentile(img: np.ndarray, p_low: float = 1.0, p_high: float = 99.0) -> np.ndarray:
    img = img.astype(np.float32)
    lo, hi = np.percentile(img, [p_low, p_high])
    if hi - lo < 1e-6:
        return np.zeros_like(img, dtype=np.float32)
    return np.clip((img - lo) / (hi - lo), 0.0, 1.0)

def load_field(row: pd.Series) -> dict:
    dapi    = tifffile.imread(TIFF_DIR / row["Image_FileName_DAPI"])
    tubulin = tifffile.imread(TIFF_DIR / row["Image_FileName_Tubulin"])
    actin   = tifffile.imread(TIFF_DIR / row["Image_FileName_Actin"])
    return {
        "dapi":    normalize_percentile(dapi),
        "tubulin": normalize_percentile(tubulin),
        "actin":   normalize_percentile(actin),
        "compound": row["compound_norm"],
        "concentration": row.get("Image_Metadata_Concentration", np.nan),
        "plate": row["Image_Metadata_Plate_DAPI"],
        "well":  row["Image_Metadata_Well_DAPI"],
    }

def make_rgb(field: dict) -> np.ndarray:
    return np.stack([field["actin"], field["tubulin"], field["dapi"]], axis=-1)

In [ ]:
# DMSO field as reference
dmso_rows = plates_meta[plates_meta["compound_norm"] == "DMSO"]
example_field = load_field(dmso_rows.iloc[0])

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(make_rgb(example_field))
axes[0].set_title("Composite (R=actin, G=tubulin, B=DAPI)")
axes[1].imshow(example_field["dapi"], cmap="Blues_r")
axes[1].set_title("DAPI (nuclei)")
axes[2].imshow(example_field["tubulin"], cmap="Greens_r")
axes[2].set_title("beta-tubulin (microtubules)")
axes[3].imshow(example_field["actin"], cmap="Reds_r")
axes[3].set_title("phalloidin (actin)")

for ax in axes:
    ax.axis("off")

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / f"dmso_reference_field_{example_field['plate']}_{example_field['well']}.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

In [ ]:
# example field for each treatment
fig, axes = plt.subplots(1, len(COMPOUND_ORDER), figsize=(16, 4))

for ax, compound in zip(axes, COMPOUND_ORDER):
    rows = plates_meta[plates_meta["compound_norm"] == compound]
    field = load_field(rows.iloc[0])
    ax.imshow(make_rgb(field))
    ax.set_title(compound)
    ax.axis("off")

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "example_field_for_each_treatment.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

In [ ]:
# one demo field for each compound for visual checks
demo_rows = []

for compound in COMPOUND_ORDER:
    rows = plates_meta[plates_meta["compound_norm"] == compound]
    if len(rows):
        demo_rows.append(rows.iloc[0])

print(f"Demo fields selected: {len(demo_rows)}")
print([row["compound_norm"] for row in demo_rows])

# Watershed baseline

**Method idea** is old; it is a heuristic invented before deep learning. The image is represented as a landscape: bright pixels are mountain peaks, dark pixels are depressions. If water is poured into each depression simultaneously, the water will rise, and the borders between basins from different depressions will give the separation into objects.

**Pipeline for MCF-7 cells:**

1. **Smoothing** both channels with `gaussian` — removing pixel noise.
2. **Otsu** on DAPI → binary nuclei mask. Otsu automatically selects a threshold that separates two peaks in the intensity histogram: background and nuclei.
3. **Distance transform** of the nuclei mask: for each pixel inside a nucleus, we calculate the distance to the nearest background pixel. The distance is maximal at the centers of nuclei.
4. **`peak_local_max`** on the distance map — coordinates of nuclei centers. These are **markers**, one per cell.
5. **Otsu on actin** → cytoplasm mask, meaning the shape of the whole cell.
6. **`watershed(-distance_actin, markers, mask=cyto_mask)`** — watershed on the inverted actin distance map, starting from the nuclei markers and restricted by the cytoplasm mask.

As a result, we get an array with the same dimensions as the original image, where each pixel has a cell ID, and 0 is the background.

The main **drawback** of watershed is very poor segmentation of cells inside clusters, stuck-together cells, and cells at the edges of the field.

Sources:

- https://scikit-image.org/docs/stable/auto_examples/segmentation/plot_watershed.html
- https://www.labellerr.com/blog/ml-guide-on-cell-segmentation-using-deep-learning/$0
- https://blog.biodock.ai/definitive-guide-to-cell-segmentation-analysis/$0
- https://www.youtube.com/watch?v=M1mJsJ5M4iE$0
- https://www.doc.ic.ac.uk/~sa1013/pub/2012_S_Arslan_PlosOne.pdf$0
- https://big-fish.readthedocs.io/en/stable/segmentation/cell.html$0

In [ ]:
def segment_watershed(field: dict, min_distance_peaks: int = 15) -> np.ndarray:
    """
    Watershed segmentation: DAPI nuclei for seed markers,
    actin for the whole-cell mask
    """
    # Smooth both fluorescence channels to reduce pixel-level noise before thresholding
    # makes Otsu thresholds more stable
    # prevents isolated bright pixels from being interpreted as nuclei
    dapi_sm   = filters.gaussian(field["dapi"],  sigma=2.0)
    actin_sm  = filters.gaussian(field["actin"], sigma=2.0)

    # Otsu thresholding to the DAPI and actin
    # DAPI to detect nuclei
    # actin to estimate broader cell region
    t_dapi  = filters.threshold_otsu(dapi_sm)
    t_actin = filters.threshold_otsu(actin_sm)
    nuclei_mask = dapi_sm > t_dapi
    cyto_mask   = actin_sm > t_actin

    # Clean nuclei mask morphologically
    # Small bright artefacts removed, small gaps inside nuclei closed
    # each nucleus is more likely to form one object
    nuclei_mask = morphology.remove_small_objects(nuclei_mask, min_size=80)
    nuclei_mask = morphology.binary_closing(nuclei_mask, morphology.disk(3))

    # Compute the distance transform of the nuclei mask and find local maxima
    # Center nucleus pixels have the largest distance from background,
    # local peaks as approximate nucleus centers
    distance = ndi.distance_transform_edt(nuclei_mask)
    coords = peak_local_max(
        distance,
        min_distance=min_distance_peaks,
        labels=nuclei_mask,
    )

    # Convert the peak coordinates into a binary peak mask, label connected peaks
    # labelled peaks become watershed markers
    peak_mask = np.zeros(distance.shape, dtype=bool)
    if len(coords):
        peak_mask[tuple(coords.T)] = True
    markers, _ = ndi.label(peak_mask)

    # Combine the actin-based cytoplasm mask with the nuclei mask
    # ensures nuclei included in the final segmentation area even when
    # the actin signal is weak or fragmented
    full_mask = cyto_mask | nuclei_mask
    full_mask = morphology.remove_small_objects(full_mask, min_size=200)

    # Run watershed on the inverted distance map of the full cell mask
    # The labelled nuclei markers define where each cell region starts growing,
    # full_mask restricts the segmentation to likely cellular pixels only
    distance_actin = ndi.distance_transform_edt(full_mask)
    labels = segmentation.watershed(-distance_actin, markers=markers, mask=full_mask)
    return labels

In [ ]:
fig, axes = plt.subplots(len(demo_rows), 4, figsize=(16, 4 * len(demo_rows)))

if len(demo_rows) == 1:
    axes = axes[np.newaxis, :]

for i, row in enumerate(demo_rows):
    f = load_field(row)
    labels = segment_watershed(f)

    composite_img = make_rgb(f)

    watershed_input = np.stack(
        [
            f["actin"],
            np.zeros_like(f["dapi"]),
            f["dapi"],
        ],
        axis=-1,
    )

    overlay = label2rgb(
        labels,
        image=watershed_input,
        bg_label=0,
        alpha=0.3,
        image_alpha=1.0,
        saturation=0.3,
    )

    axes[i, 0].imshow(watershed_input)
    axes[i, 0].set_title("Watershed input: actin + DAPI")
    axes[i, 1].imshow(label2rgb(labels, bg_label=0))
    axes[i, 1].set_title(f"Watershed mask: {labels.max()} cells")
    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title("Mask on input")
    axes[i, 3].imshow(composite_img)
    axes[i, 3].set_title(f"{f['compound']}: 3-channel composite")

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "naive_watershed.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

Watershed segments isolated single cells very well, but those are the minority in our images; for touching or clustered cells, it is essentially useless. This is a fundamental problem of threshold-based methods: they do not understand what a cell is; they simply separate bright regions from dark ones.

# StarDist

StarDist is a deep-learning method for instance segmentation. Unfortunately, it is not suitable for whole cells, but it segments nuclei very well. Therefore, we use it for two things:

1) to count the number of cells independently of Cellpose and compare the counts, to check that the order of magnitude is roughly the same;

2) to detect cell nuclei and try passing them as markers to the watershed method, then see what happens.

In [ ]:
stardist_model = StarDist2D.from_pretrained("2D_versatile_fluo")

def segment_stardist_nuclei(field: dict) -> np.ndarray:
    labels, details = stardist_model.predict_instances(normalize(field["dapi"]))
    return labels

In [ ]:
# StarDist nuclei
fig, axes = plt.subplots(len(demo_rows), 4, figsize=(16, 4 * len(demo_rows)))

if len(demo_rows) == 1:
    axes = axes[np.newaxis, :]

for i, row in enumerate(demo_rows):
    f = load_field(row)
    labels = segment_stardist_nuclei(f)

    composite_img = make_rgb(f)

    stardist_input = f["dapi"]

    overlay = label2rgb(
        labels,
        image=stardist_input,
        bg_label=0,
        alpha=0.3,
        image_alpha=1.0,
        saturation=0.3,
    )

    axes[i, 0].imshow(stardist_input, cmap="Blues_r", vmin=0, vmax=1)
    axes[i, 0].set_title("StarDist input: DAPI")

    axes[i, 1].imshow(label2rgb(labels, bg_label=0))
    axes[i, 1].set_title(f"StarDist mask: {labels.max()} nuclei")

    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title("StarDist mask on DAPI")

    axes[i, 3].imshow(composite_img)
    axes[i, 3].set_title(f"{f['compound']}: 3-channel composite")

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "stardist_nuclei.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

Combine StarDist and watershed to see whether more accurate detection of cell nuclei improves segmentation.

StarDist detects nuclei on the DAPI channel and uses them as markers, while actin + tubulin define the cell region. Watershed then expands each nucleus into the cell mask, with pixels assigned to the nearest markers up to the mask boundary, but not beyond the nearest detected nucleus in each direction.

In essence, this is marker-controlled watershed. The nuclei are the markers, and watershed distributes the allowed region between them.

In [ ]:
def starshed(field: dict, prob_thresh: float = 0.35, nms_thresh: float = 0.4) -> np.ndarray:
    """
    StarDist detects nuclei from DAPI
    Watershed expands each nucleus within a whole-cell mask from actin + tubulin
    """
    # StarDist returns an instance-labelled image
    # 0 is background, 1..N are detected nuclei
    nuclei_labels, details = stardist_model.predict_instances(
        normalize(field["dapi"]),
        prob_thresh=prob_thresh,
        nms_thresh=nms_thresh,
    )
    markers = nuclei_labels.astype(np.int32)

    # Build the whole-cell signal from actin + tubulin
    # Each channel is smoothed, then normalized
    actin_sm   = filters.gaussian(field["actin"],   sigma=2.0)
    tubulin_sm = filters.gaussian(field["tubulin"], sigma=2.0)

    cell_signal = np.maximum(
        normalize(actin_sm),
        normalize(tubulin_sm),
    )

    # Otsu thresholding to combined cytoskeleton signal
    # creates the approximate area of whole cells
    t_cell = filters.threshold_otsu(cell_signal)
    cell_mask = cell_signal > t_cell

    # include all detected nuclei in the mask
    # then remove small disconnected objects
    cell_mask = cell_mask | (markers > 0)
    cell_mask = morphology.remove_small_objects(cell_mask, min_size=200)

    # Run marker-controlled watershed inside the allowed cell mask
    # StarDist nuclei define the starting regions, and watershed assigns
    # the remaining cell-mask pixels to the nearest marker
    distance = ndi.distance_transform_edt(cell_mask)
    labels = segmentation.watershed(
        -distance,
        markers=markers,
        mask=cell_mask,
    )

    return labels

In [ ]:
# StarDist nuclei + watershed
fig, axes = plt.subplots(len(demo_rows), 4, figsize=(16, 4 * len(demo_rows)))

if len(demo_rows) == 1:
    axes = axes[np.newaxis, :]

for i, row in enumerate(demo_rows):
    f = load_field(row)

    composite_img = make_rgb(f)

    starshed_input = np.stack(
        [
            f["actin"],
            f["tubulin"],
            f["dapi"],
        ],
        axis=-1,
    )

    labels = starshed(f)

    overlay = label2rgb(
        labels,
        image=starshed_input,
        bg_label=0,
        alpha=0.3,
        image_alpha=1.0,
        saturation=0.3,
    )

    axes[i, 0].imshow(starshed_input)
    axes[i, 0].set_title("Input: actin + DAPI")

    axes[i, 1].imshow(label2rgb(labels, bg_label=0))
    axes[i, 1].set_title(f"StarDist + watershed mask: {labels.max()} cells")

    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title("Mask on input")

    axes[i, 3].imshow(composite_img)
    axes[i, 3].set_title(f"{f['compound']}: 3-channel composite")

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "starshed_cells.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

# Cellpose

[Cellpose](https://cellpose.readthedocs.io/) is a family of convolutional neural networks trained on tens of thousands of manually annotated cells of different types. The **`cyto3`** model is a general-purpose version that works across a wide range of cell types without further training.

**The idea of Cellpose**: the network predicts a **vector field**, that is, for each pixel, the direction toward the center of its own cell. These vectors are then integrated: pixels whose trajectories converge to the same point end up belonging to the same cell. This solves the main problem of watershed: touching cells are separated because their centers are different, and the vectors from them point in different directions.

Cellpose `cyto3` expects a two-channel image as input. For BBBC021, this is:
- **cytoplasm** — actin, which shows the overall cell shape;
- **nuclei** — DAPI.

That is, the model input is `channels=[1, 3]`.

The first model load downloads the weights (25 MB) to `~/.cellpose/models/`.

In [ ]:
# download Cellpose 3 weights
cp_model = cp_models.Cellpose(model_type="cyto3", gpu=torch.cuda.is_available())
print(f"Cellpose готов. GPU: {cp_model.gpu}")

In [ ]:
def prepare_cellpose_input(field: dict) -> np.ndarray:
    """Forms Cellpose input (H, W, 3): R=актин, G=none, B=DAPI"""
    return np.stack(
        [
            field["actin"],
            np.zeros_like(field["dapi"]),
            field["dapi"],
        ],
        axis=-1,
    )

def segment_cellpose(field: dict, diameter: float | None = None) -> np.ndarray:
    """Runs cellpose on a field and returns mask with cells IDs"""
    img = prepare_cellpose_input(field)
    out = cp_model.eval(img, diameter=diameter, channels=[1, 3])
    masks = out[0]
    return masks.astype(np.int32)

# For main pipeline, only need masks segment_cellpose returns
# masks is a 2D array where 0 = background and 1, 2, 3... = individual cell IDs

# For inspecting the model, segment_cellpose_inside returns:
# flows - vector fields and related maps predicted by Cellpose
# diams - cell diameter that Cellpose used or estimated

def segment_cellpose_inside(field: dict, diameter: float | None = None):
    """Runs Cellpose and returns masks, flows, diams"""
    img = prepare_cellpose_input(field)
    masks, flows, _styles, diams = cp_model.eval(img, diameter=diameter, channels=[1, 3])
    return masks.astype(np.int32), flows, diams

In [ ]:
# segment_cellpose test
fig, axes = plt.subplots(len(demo_rows), 4, figsize=(16, 4 * len(demo_rows)))

if len(demo_rows) == 1:
    axes = axes[np.newaxis, :]

for i, row in enumerate(demo_rows):
    f = load_field(row)

    composite_img = make_rgb(f)
    model_input = prepare_cellpose_input(f)
    labels = segment_cellpose(f, diameter=None)

    overlay = label2rgb(
        labels,
        image=model_input,
        bg_label=0,
        alpha=0.3,
        image_alpha=1.0,
        saturation=0.3,
    )

    axes[i, 0].imshow(model_input)
    axes[i, 0].set_title("Cellpose input: actin + DAPI")

    axes[i, 1].imshow(label2rgb(labels, bg_label=0))
    axes[i, 1].set_title(f"Cellpose mask: {labels.max()} cells")

    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title("Mask over input")

    axes[i, 3].imshow(composite_img)
    axes[i, 3].set_title(f"{f['compound']}: 3-channel composite")

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "cellpose_baseline_test.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

In [ ]:
# segment_cellpose_inside test

fig, axes = plt.subplots(len(demo_rows), 5, figsize=(20, 4 * len(demo_rows)), squeeze=False)

for i, row in enumerate(demo_rows):
    f = load_field(row)

    composite_img = make_rgb(f)
    model_input = prepare_cellpose_input(f)

    labels, flows, diams = segment_cellpose_inside(f, diameter=None)

    overlay = label2rgb(
        labels,
        image=model_input,
        bg_label=0,
        alpha=0.3,
        image_alpha=1.0,
        saturation=0.3,
    )

    flow_rgb = flows[0]
    cellprob = flows[2]

    axes[i, 0].imshow(model_input)
    axes[i, 0].set_title("Cellpose input: actin + DAPI")

    axes[i, 1].imshow(label2rgb(labels, bg_label=0))
    axes[i, 1].set_title(f"Cellpose mask: {labels.max()} cells")

    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title("Mask over input")

    axes[i, 3].imshow(flow_rgb)
    axes[i, 3].set_title("Cellpose flow field")

    im = axes[i, 4].imshow(cellprob, cmap="magma")
    axes[i, 4].set_title(f"Cell probability; diameter={diams:.1f}")

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "cellpose_flows_and_cellprob.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

# Watershed, StarDist and Cellpose comparison



Compare all three approaches simultaneously across all compounds.

In [ ]:
fig, axes = plt.subplots(len(demo_rows), 3, figsize=(15, 4 * len(demo_rows)))

if len(demo_rows) == 1:
    axes = axes[np.newaxis, :]

for i, row in enumerate(demo_rows):
    f = load_field(row)
    composite_img = make_rgb(f)

    watershed_labels = segment_watershed(f)
    stardist_watershed_labels = starshed(f)
    cellpose_labels = segment_cellpose(f, diameter=None)

    watershed_overlay = label2rgb(
        watershed_labels,
        image=composite_img,
        bg_label=0,
        alpha=0.3,
        image_alpha=1.0,
        saturation=0.3,
    )

    stardist_watershed_overlay = label2rgb(
        stardist_watershed_labels,
        image=composite_img,
        bg_label=0,
        alpha=0.3,
        image_alpha=1.0,
        saturation=0.3,
    )

    cellpose_overlay = label2rgb(
        cellpose_labels,
        image=composite_img,
        bg_label=0,
        alpha=0.3,
        image_alpha=1.0,
        saturation=0.3,
    )

    axes[i, 0].imshow(watershed_overlay)
    axes[i, 0].set_title(f"{f['compound']}\nWatershed: {watershed_labels.max()} cells")

    axes[i, 1].imshow(stardist_watershed_overlay)
    axes[i, 1].set_title(f"{f['compound']}\nStarDist + watershed: {stardist_watershed_labels.max()} cells")

    axes[i, 2].imshow(cellpose_overlay)
    axes[i, 2].set_title(f"{f['compound']}\nCellpose: {cellpose_labels.max()} cells")

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "watershed_stardist_cellpose.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

# StarDist and Cellpose on nuclei

Since we do not have ground truth for cells, we will try to indirectly check whether Cellpose and StarDist at least detect cell nuclei comparably. This is not a full validation, because it does not prove that the masks are equally good. After all, for whole-cell segmentation we provide different input channels. Still, a similar number of objects would mean that Cellpose nuclei does **not fail completely** in terms of count relative to StarDist.

In [ ]:
# Compare nuclei segmentation: StarDist nuclei vs Cellpose nuclei

cp_nuclei_model = cp_models.Cellpose(
    model_type="nuclei",
    gpu=torch.cuda.is_available(),
)

print(f"Cellpose nuclei ready. GPU: {cp_nuclei_model.gpu}")


def segment_cellpose_nuclei(field: dict, diameter: float | None = None) -> np.ndarray:
    """Runs Cellpose nuclei model on DAPI and returns mask with nucleus IDs"""
    dapi = field["dapi"]

    out = cp_nuclei_model.eval(
        dapi,
        diameter=diameter,
        channels=[0, 0],
    )

    masks = out[0]
    return masks.astype(np.int32)


def nuclei_method_mask_path(row: pd.Series, method: str) -> Path:
    """Cache path for nuclei segmentation masks."""
    stem = Path(row["Image_FileName_DAPI"]).stem
    return MASKS_DIR / f"{stem}_{method}.npy"


nuclei_methods = {
    "stardist_nuclei": segment_stardist_nuclei,
    "cellpose_nuclei": segment_cellpose_nuclei,
}

nuclei_counts = {}

for method_name, segment_func in nuclei_methods.items():
    masks_by_idx = {}

    for idx, row in tqdm(
        plates_meta.iterrows(),
        total=len(plates_meta),
        desc=method_name,
    ):
        cache_path = nuclei_method_mask_path(row, method_name)

        if cache_path.exists():
            masks = np.load(cache_path)
        else:
            field = load_field(row)
            masks = segment_func(field)
            np.save(cache_path, masks)

        masks_by_idx[idx] = masks

    count_col = f"n_{method_name}"
    plates_meta[count_col] = plates_meta.index.map(
        pd.Series({idx: int(m.max()) for idx, m in masks_by_idx.items()})
    )

    nuclei_counts[method_name] = masks_by_idx

In [ ]:
summary = (
    plates_meta
    .groupby("compound_norm")[["n_stardist_nuclei", "n_cellpose_nuclei"]]
    .mean()
    .reindex(COMPOUND_ORDER)
)

display(summary.round(1))

In [ ]:
ax = summary.plot(
    kind="bar",
    figsize=(10, 5),
    width=0.75,
)

ax.set_xlabel("Compound")
ax.set_ylabel("Mean nuclei per field")
ax.legend(["StarDist nuclei", "Cellpose nuclei"])

plt.xticks(rotation=30, ha="right")
plt.tight_layout()

fig = ax.get_figure()
fig.savefig(
    FIGURES_DIR / "nuclei_count_stardist_vs_cellpose.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

In [ ]:
# AZ258 nuclei segmentation comparison StarDist / Cellpose

az258_rows = plates_meta[
    plates_meta["compound_norm"] == "AZ258"
].copy()

az258_rows["nuclei_count_difference"] = (
    az258_rows["n_stardist_nuclei"] - az258_rows["n_cellpose_nuclei"]
).abs()

az258_rows = (
    az258_rows
    .sort_values("nuclei_count_difference", ascending=False)
    .head(4)
)

display(
    az258_rows[
        [
            "compound_norm",
            "Image_Metadata_Concentration",
            "Image_Metadata_Plate_DAPI",
            "Image_Metadata_Well_DAPI",
            "n_stardist_nuclei",
            "n_cellpose_nuclei",
            "nuclei_count_difference",
        ]
    ]
)

In [ ]:
fig, axes = plt.subplots(
    len(az258_rows),
    2,
    figsize=(8, 4 * len(az258_rows)),
    squeeze=False,
)

for row_i, (idx, row) in enumerate(az258_rows.iterrows()):
    field = load_field(row)
    dapi = field["dapi"]

    stardist_path = nuclei_method_mask_path(row, "stardist_nuclei")
    cellpose_path = nuclei_method_mask_path(row, "cellpose_nuclei")

    stardist_mask = np.load(stardist_path)
    cellpose_mask = np.load(cellpose_path)

    stardist_fg = stardist_mask > 0
    cellpose_fg = cellpose_mask > 0

    overlap = stardist_fg & cellpose_fg
    stardist_only = stardist_fg & ~cellpose_fg
    cellpose_only = cellpose_fg & ~stardist_fg

    dapi_rgb = np.stack([dapi, dapi, dapi], axis=-1)

    diff_overlay = dapi_rgb.copy()
    diff_overlay[overlap] = (
        0.55 * diff_overlay[overlap]
        + 0.45 * np.array([1.0, 1.0, 0.0])
    )
    diff_overlay[stardist_only] = (
        0.35 * diff_overlay[stardist_only]
        + 0.65 * np.array([1.0, 0.25, 0.65])
    )
    diff_overlay[cellpose_only] = (
        0.35 * diff_overlay[cellpose_only]
        + 0.65 * np.array([0.0, 0.8, 1.0])
    )
    diff_overlay = np.clip(diff_overlay, 0.0, 1.0)

    title_prefix = (
        f"AZ258, conc={row['Image_Metadata_Concentration']:g}, "
        f"well={row['Image_Metadata_Well_DAPI']}"
    )

    axes[row_i, 0].imshow(diff_overlay)
    axes[row_i, 0].set_title(
        f"{title_prefix}\n"
        "pink=StarDist, green=Cellpose, yel=both"
    )

    axes[row_i, 1].imshow(dapi, cmap="gray", vmin=0, vmax=1)
    axes[row_i, 1].set_title(
        f"DAPI\n"
        f"StarDist={int(stardist_mask.max())}, "
        f"Cellpose={int(cellpose_mask.max())}"
    )

    for ax in axes[row_i]:
        ax.axis("off")

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "az258_stardist_cellpose_nuclei_diff.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

# Improving Cellpose segmentation

As the mask overlays show, Cellpose segments well, but not perfectly. Some cells are merged, and some overly large cells are split into several objects. Compared with the other methods, this is not a major issue, but we will still try to improve the segmentation.

Fine-tuning the model would be slow and expensive. Therefore, we will try to make the **input** more complete and higher-contrast. The methods are:

1. **Merge actin and tubulin into one cytoplasm channel** using `np.maximum`. Biologically, both proteins occupy the cytoplasm; when one is depolymerized, the other can compensate. Logically, this acts as an “OR” between the two signals.

2. **CLAHE** (Contrast-Limited Adaptive Histogram Equalization) — a local contrast enhancer. It divides the image into tiles and normalizes the contrast within each tile separately, while limiting amplification so that noise is not excessively increased. This is a standard technique in medical imaging, including ultrasound, X-ray, and histology.

3. **Adjust Cellpose hyperparameters** — specify the expected cell diameter and lower the confidence threshold `cellprob_threshold`, so that the network is more willing to detect dim cells. Increase the `flow_threshold` so that Cellpose keeps more masks.

All three methods can be additive.

# Cellpose hyperparameter tuning

According to the Cellpose v3.1 documentation, the default parameters are:
- `diameter=None` enables auto-estimation
- `flow_threshold=0.4`
- `cellprob_threshold=0.0`

Sources:
- https://cellpose.readthedocs.io/en/v3.1.1.1/settings.html
- https://cellpose.readthedocs.io/en/v3.1.1.1/api.html

Hyperparameter description:

1. `diameter`, in pixels

  - `diameter = None/0` — Cellpose estimates the object size automatically.  
  - `diameter > 0` — manually specified expected object diameter in pixels.

There is no universal range. It depends on the image scale. In practice, `diameter` is often set around 20–100 px. If `diameter` is set smaller than the real cell size, Cellpose may split one cell into several objects. If `diameter` is set larger than the real cell size, Cellpose may merge neighboring cells.

2. `cellprob_threshold`

Default: `0.0`.

This controls the separation of objects from the background. Cellpose states that cell probability logits are approximately in the range from `-6` to `+6`. In practice, a useful range is often around `-3.0` to `+3.0`.

- Below `0.0` — more permissive, more pixels/masks. Too low — risk of debris and over-segmentation.
- Above `0.0` — stricter, fewer masks. Too high — risk of missing dim cells.

3. `flow_threshold`

Default: `0.4`. `0` disables this filter. In practice, a commonly used range is around `0.1–3.0`.

- Below `0.4` — stricter: Cellpose discards more questionable masks.  
- Above `0.4` — more permissive: Cellpose keeps more ROIs, but may also keep distorted masks. Too high — risk of accepting poor masks.

In [ ]:
BASELINE_CELLPOSE_PARAMS = dict(
    diameter=None,
    channels=[1, 3],
)

def segment_cellpose(field: dict, **cellpose_params) -> np.ndarray:
    """Runs cellpose on a field and returns mask with cells IDs"""
    img = prepare_cellpose_input(field)
    params = BASELINE_CELLPOSE_PARAMS.copy()
    params.update(cellpose_params)
    out = cp_model.eval(img, **params)
    return out[0].astype(np.int32)

def cellpose_variant_mask_path(row: pd.Series, variant: str) -> Path:
    stem = Path(row["Image_FileName_DAPI"]).stem
    return MASKS_DIR / f"{stem}__cellpose_{variant}.npy"

def overlay_outlines_simple(
    rgb: np.ndarray,
    masks: np.ndarray,
    color=(1.0, 1.0, 0.0),
    thickness: int = 1,
) -> np.ndarray:
    """lays mask boarders over RGB-composite"""
    out = rgb.copy()
    boundaries = find_boundaries(masks, mode="outer")

    if thickness > 1:
        boundaries = morphology.binary_dilation(boundaries, morphology.disk(thickness))

    for c in range(3):
        out[..., c] = np.where(boundaries, color[c], out[..., c])

    return np.clip(out, 0.0, 1.0)

# Run grid search

In [ ]:
GRID_DIAMETERS = [None, 50, 70, 90]
GRID_CELLPROB_THRESHOLDS = [1.0, -0.5, -1.0]
GRID_FLOW_THRESHOLDS = [0.2, 0.6, 1.0]

CELLPOSE_GRID_VARIANTS = {
    "grid_baseline": {
        "label": "baseline: auto / cp=0.0 / flow=0.4",
        "params": BASELINE_CELLPOSE_PARAMS,
        "count_col": "n_cells_cellpose",
    }
}

for diameter, cellprob_threshold, flow_threshold in product(
    GRID_DIAMETERS,
    GRID_CELLPROB_THRESHOLDS,
    GRID_FLOW_THRESHOLDS,
):
    if diameter is None and cellprob_threshold == 0.0 and flow_threshold == 0.4:
        continue

    diameter_label = "dauto" if diameter is None else f"d{diameter}"
    cellprob_label = str(cellprob_threshold).replace("-", "m").replace(".", "p")
    flow_label = str(flow_threshold).replace(".", "p")

    variant = f"grid_{diameter_label}_cp{cellprob_label}_f{flow_label}"

    CELLPOSE_GRID_VARIANTS[variant] = {
        "label": f"d={diameter}, cp={cellprob_threshold}, flow={flow_threshold}",
        "params": dict(
            diameter=diameter,
            cellprob_threshold=cellprob_threshold,
            flow_threshold=flow_threshold,
            channels=[1, 3],
        ),
        "count_col": f"n_cells_{variant}",
    }

for variant, spec in CELLPOSE_GRID_VARIANTS.items():
    masks_by_idx = {}

    for idx, row in tqdm(
        plates_meta.iterrows(),
        total=len(plates_meta),
        desc=f"Cellpose {variant}",
    ):
        cache_path = cellpose_variant_mask_path(row, variant)

        if cache_path.exists():
            masks = np.load(cache_path)
        else:
            field = load_field(row)
            masks = segment_cellpose(field, **spec["params"])
            np.save(cache_path, masks)

        masks_by_idx[idx] = masks

    plates_meta[spec["count_col"]] = plates_meta.index.map(
        pd.Series({idx: int(m.max()) for idx, m in masks_by_idx.items()})
    )

In [ ]:
cellpose_grid_variants_df = pd.DataFrame([
    {
        "variant": variant,
        "label": spec["label"],
        "count_col": spec["count_col"],
    }
    for variant, spec in CELLPOSE_GRID_VARIANTS.items()
])

display(cellpose_grid_variants_df)

In [ ]:
# visual check of grid search

GRID_VARIANTS_TO_SHOW = cellpose_grid_variants_df["variant"].tolist()

fig, axes = plt.subplots(
    len(GRID_VARIANTS_TO_SHOW),
    len(demo_rows),
    figsize=(4 * len(demo_rows), 3.2 * len(GRID_VARIANTS_TO_SHOW)),
)

if len(GRID_VARIANTS_TO_SHOW) == 1:
    axes = axes[np.newaxis, :]

if len(demo_rows) == 1:
    axes = axes[:, np.newaxis]

for i, variant in enumerate(GRID_VARIANTS_TO_SHOW):
    variant_label = CELLPOSE_GRID_VARIANTS[variant]["label"]

    for j, row in enumerate(demo_rows):
        field = load_field(row)
        composite_img = make_rgb(field)

        masks = np.load(cellpose_variant_mask_path(row, variant))
        overlay = overlay_outlines_simple(composite_img, masks, thickness=1)

        axes[i, j].imshow(overlay)
        axes[i, j].set_title(
            f"{field['compound']}\n{variant_label}\n{masks.max()} cells",
            fontsize=8,
        )
        axes[i, j].axis("off")

plt.tight_layout()
fig.savefig(
    FIGURES_DIR / "cellpose_grid_search_overlays.pdf",
    bbox_inches="tight",
    dpi=300,
)
plt.show()

In [ ]:
# Change in segmented objects
grid_growth_records = []

for variant, spec in CELLPOSE_GRID_VARIANTS.items():
    if variant == "grid_baseline":
        continue
    params = spec["params"]

    diameter = params.get("diameter", None)
    cellprob_threshold = params.get("cellprob_threshold", 0.0)
    flow_threshold = params.get("flow_threshold", 0.4)
    count_col = spec["count_col"]

    for compound, g in plates_meta.groupby("compound_norm"):
        baseline_mean = g["n_cells_cellpose"].mean()
        variant_mean = g[count_col].mean()

        growth_percent = 100 * (variant_mean - baseline_mean) / baseline_mean

        grid_growth_records.append({
            "variant": variant,
            "diameter": "auto" if diameter is None else str(int(diameter)),
            "cellprob_threshold": cellprob_threshold,
            "flow_threshold": flow_threshold,
            "parameter_label": f"cp={cellprob_threshold}, flow={flow_threshold}",
            "compound_norm": compound,
            "growth_percent": growth_percent,
        })

grid_growth_df = pd.DataFrame(grid_growth_records)

diameter_order = ["auto", "50", "70", "90"]
parameter_order = (
    grid_growth_df[["cellprob_threshold", "flow_threshold", "parameter_label"]]
    .drop_duplicates()
    .sort_values(["cellprob_threshold", "flow_threshold"])
    ["parameter_label"]
    .tolist()
)

fig, axes = plt.subplots(
    len(diameter_order),
    1,
    figsize=(16, 5 * len(diameter_order)),
    sharey=True,
)

for ax, diameter_label in zip(axes, diameter_order):
    plot_df = grid_growth_df[
        grid_growth_df["diameter"] == diameter_label
    ].copy()

    sns.barplot(
        data=plot_df,
        x="compound_norm",
        y="growth_percent",
        hue="parameter_label",
        order=COMPOUND_ORDER,
        hue_order=parameter_order,
        ax=ax,
    )

    ax.axhline(0, color="black", linewidth=1)
    ax.set_title(f"Cellpose diameter = {diameter_label}")
    ax.set_ylabel("Cell count relative to baseline, %")
    ax.set_xlabel("")
    ax.grid(True, axis="y", alpha=0.3)

    ax.legend(
        title="Parameters",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )

plt.tight_layout()
fig.savefig(
    FIGURES_DIR / "cellpose_grid_search_cell_count_growth.pdf",
    bbox_inches="tight",
    dpi=300,
)
plt.show()

In [ ]:
# grid search with diameter = auto
GRID_DIAMETERS = [None]
GRID_CELLPROB_THRESHOLDS = [-2.0, -1.0, 0.0, 1.0, 2.0]
GRID_FLOW_THRESHOLDS = [0.4, 0.6, 0.8, 1.0]

CELLPOSE_GRID_VARIANTS = {
    "grid_baseline": {
        "label": "baseline: auto / cp=0.0 / flow=0.4",
        "params": BASELINE_CELLPOSE_PARAMS,
        "count_col": "n_cells_cellpose",
    }
}

for diameter, cellprob_threshold, flow_threshold in product(
    GRID_DIAMETERS,
    GRID_CELLPROB_THRESHOLDS,
    GRID_FLOW_THRESHOLDS,
):
    if diameter is None and cellprob_threshold == 0.0 and flow_threshold == 0.4:
        continue

    diameter_label = "dauto" if diameter is None else f"d{diameter}"
    cellprob_label = str(cellprob_threshold).replace("-", "m").replace(".", "p")
    flow_label = str(flow_threshold).replace(".", "p")

    variant = f"grid_{diameter_label}_cp{cellprob_label}_f{flow_label}"

    CELLPOSE_GRID_VARIANTS[variant] = {
        "label": f"d={diameter}, cp={cellprob_threshold}, flow={flow_threshold}",
        "params": dict(
            diameter=diameter,
            cellprob_threshold=cellprob_threshold,
            flow_threshold=flow_threshold,
            channels=[1, 3],
        ),
        "count_col": f"n_cells_{variant}",
    }

for variant, spec in CELLPOSE_GRID_VARIANTS.items():
    masks_by_idx = {}

    for idx, row in tqdm(
        plates_meta.iterrows(),
        total=len(plates_meta),
        desc=f"Cellpose {variant}",
    ):
        cache_path = cellpose_variant_mask_path(row, variant)

        if cache_path.exists():
            masks = np.load(cache_path)
        else:
            field = load_field(row)
            masks = segment_cellpose(field, **spec["params"])
            np.save(cache_path, masks)

        masks_by_idx[idx] = masks

    plates_meta[spec["count_col"]] = plates_meta.index.map(
        pd.Series({idx: int(m.max()) for idx, m in masks_by_idx.items()})
    )

In [ ]:
cellpose_grid_variants_df = pd.DataFrame([
    {
        "variant": variant,
        "label": spec["label"],
        "count_col": spec["count_col"],
    }
    for variant, spec in CELLPOSE_GRID_VARIANTS.items()
])

display(cellpose_grid_variants_df)

In [ ]:
# visual check of grid search
GRID_VARIANTS_TO_SHOW = cellpose_grid_variants_df["variant"].tolist()

fig, axes = plt.subplots(
    len(GRID_VARIANTS_TO_SHOW),
    len(demo_rows),
    figsize=(4 * len(demo_rows), 3.2 * len(GRID_VARIANTS_TO_SHOW)),
)

if len(GRID_VARIANTS_TO_SHOW) == 1:
    axes = axes[np.newaxis, :]

if len(demo_rows) == 1:
    axes = axes[:, np.newaxis]

for i, variant in enumerate(GRID_VARIANTS_TO_SHOW):
    variant_label = CELLPOSE_GRID_VARIANTS[variant]["label"]

    for j, row in enumerate(demo_rows):
        field = load_field(row)
        composite_img = make_rgb(field)

        masks = np.load(cellpose_variant_mask_path(row, variant))
        overlay = overlay_outlines_simple(composite_img, masks, thickness=1)

        axes[i, j].imshow(overlay)
        axes[i, j].set_title(
            f"{field['compound']}\n{variant_label}\n{masks.max()} cells",
            fontsize=8,
        )
        axes[i, j].axis("off")

plt.tight_layout()
fig.savefig(
    FIGURES_DIR / "cellpose_grid_search_overlays_auto.pdf",
    bbox_inches="tight",
    dpi=300,
)
plt.show()

In [ ]:
# change in segmented objects
grid_growth_records = []

for variant, spec in CELLPOSE_GRID_VARIANTS.items():
    if variant == "grid_baseline":
        continue
    params = spec["params"]

    diameter = params.get("diameter", None)
    cellprob_threshold = params.get("cellprob_threshold", 0.0)
    flow_threshold = params.get("flow_threshold", 0.4)
    count_col = spec["count_col"]

    for compound, g in plates_meta.groupby("compound_norm"):
        baseline_mean = g["n_cells_cellpose"].mean()
        variant_mean = g[count_col].mean()

        growth_percent = 100 * (variant_mean - baseline_mean) / baseline_mean

        grid_growth_records.append({
            "variant": variant,
            "diameter": "auto" if diameter is None else str(int(diameter)),
            "cellprob_threshold": cellprob_threshold,
            "flow_threshold": flow_threshold,
            "parameter_label": f"cp={cellprob_threshold}, flow={flow_threshold}",
            "compound_norm": compound,
            "growth_percent": growth_percent,
        })

grid_growth_df = pd.DataFrame(grid_growth_records)

diameter_order = ["auto"]
parameter_order = (
    grid_growth_df[["cellprob_threshold", "flow_threshold", "parameter_label"]]
    .drop_duplicates()
    .sort_values(["cellprob_threshold", "flow_threshold"])
    ["parameter_label"]
    .tolist()
)

fig, axes = plt.subplots(
    len(diameter_order),
    1,
    figsize=(16, 5 * len(diameter_order)),
    sharey=True,
    squeeze=False,
)

for ax, diameter_label in zip(axes.ravel(), diameter_order):
    plot_df = grid_growth_df[
        grid_growth_df["diameter"] == diameter_label
    ].copy()

    sns.barplot(
        data=plot_df,
        x="compound_norm",
        y="growth_percent",
        hue="parameter_label",
        order=COMPOUND_ORDER,
        hue_order=parameter_order,
        ax=ax,
    )

    ax.axhline(0, color="black", linewidth=1)
    ax.set_title(f"Cellpose diameter = {diameter_label}")
    ax.set_ylabel("Cell count relative to baseline, %")
    ax.set_xlabel("")
    ax.grid(True, axis="y", alpha=0.3)

    ax.legend(
        title="Parameters",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )

plt.tight_layout()
fig.savefig(
    FIGURES_DIR / "cellpose_grid_search_cell_count_growth_auto.pdf",
    bbox_inches="tight",
    dpi=300,
)
plt.show()

We choose hyperparameters that produce stable, visually plausible masks across all compound groups, even if the cell count is not maximal.

# CLAHE and merging actin + tubulin

`equalize_adapthist` takes an image in the `[0, 1]` range and returns an image in the `[0, 1]` range with locally enhanced contrast. Parameters:

- `clip_limit=0.01` — limits the maximum contrast amplification within each tile; a low value gives mild equalization.
- `kernel_size=512` — the size of the local window, equal to 1/2 of 1024. The smaller the window, the more aggressive the local normalization.

After applying CLAHE to both channels, we take the pixel-wise maximum. This gives a merged cytoplasmic signal: in each pixel, we use the channel where the signal is stronger.

However, CLAHE is not a cure-all: it can amplify noise. Therefore, this is a hypothesis that needs to be checked with overlay images. Because of CLAHE, Cellpose may start detecting cell-like structures in regions that were only weak texture or background. CLAHE can also amplify actin/tubulin artefacts: fibers, debris, bright edges, and local spots can become too important. In addition, pretrained Cellpose has its own internal normalization, and applying CLAHE before it may make the image less similar to the training distribution. Finally, CLAHE **creates locally different contrast**: the same cell may look different in different parts of the image.

The main CLAHE parameters are:

- `clip_limit` — the strength of contrast limiting. The larger the `clip_limit`, the more aggressive the enhancement. Typical values to try are `0.001`, `0.003`, `0.005`, and `0.01`.
- `kernel_size` — the size of the local window. The smaller the window, the stronger the local normalization. For a `1024 x 1280` image, values from `128` to `512` are reasonable, but `768` or `1024` can also be tested.

In [ ]:
def clahe(
    img: np.ndarray,
    clip_limit: float = 0.01,
    kernel_size: int = 512,
) -> np.ndarray:
    """CLAHE on a single-channel float image in [0, 1]"""
    return exposure.equalize_adapthist(
        np.clip(img, 0.0, 1.0),
        clip_limit=clip_limit,
        kernel_size=kernel_size,
    ).astype(np.float32)

def merged_channels(field: dict) -> np.ndarray:
    """Merge cytoplasm channel without CLAHE"""
    return np.maximum(field["actin"], field["tubulin"]).astype(np.float32)

def clache_merged_channels(field: dict) -> np.ndarray:
    """Merge cytoplasm channel with CLAHE: CLAHE(max(actin, tubulin))"""
    merged = merged_channels(field)
    return clahe(merged, clip_limit=0.01, kernel_size=512)

In [ ]:
# actin before and after CLAHE

row = plates_meta.sample(n=1, random_state=SEED).iloc[0]
field = load_field(row)

a_raw = field["actin"]
a_clahe = clahe(a_raw)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(a_raw, cmap="Reds_r", vmin=0, vmax=1)
axes[0].set_title("Actin before CLAHE")

axes[1].imshow(a_clahe, cmap="Reds_r", vmin=0, vmax=1)
axes[1].set_title("Actin after CLAHE")

for ax in axes:
    ax.axis("off")

plt.suptitle(
    f"{row['compound_norm']} / {row['Image_Metadata_Plate_DAPI']} / {row['Image_Metadata_Well_DAPI']}",
    fontsize=12,
)

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "clahe_actin_before_after.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

In [ ]:
# merged actin + tubulin

row = plates_meta.sample(n=1, random_state=SEED).iloc[0]
field = load_field(row)

actin = field["actin"]
tubulin = field["tubulin"]
cyto_merged = merged_channels(field)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(actin, cmap="Reds_r", vmin=0, vmax=1)
axes[0].set_title("Actin")

axes[1].imshow(tubulin, cmap="Greens_r", vmin=0, vmax=1)
axes[1].set_title("Tubulin")

axes[2].imshow(cyto_merged, cmap="magma", vmin=0, vmax=1)
axes[2].set_title("max(actin, tubulin)")

for ax in axes:
    ax.axis("off")

plt.suptitle(
    f"{row['compound_norm']} / {row['Image_Metadata_Plate_DAPI']} / {row['Image_Metadata_Well_DAPI']}",
    fontsize=12,
)

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "merged_actin_tubulin.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

In [ ]:
# actin + tubulin + CLAHE

row = plates_meta.sample(n=1, random_state=SEED).iloc[0]
field = load_field(row)

actin = field["actin"]
tubulin = field["tubulin"]

cyto_raw = merged_channels(field) # max(actin, tubulin)
clahe_both = clahe(cyto_raw)     # CLAHE

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(actin, cmap="Reds_r", vmin=0, vmax=1)
axes[0].set_title("Actin")

axes[1].imshow(tubulin, cmap="Greens_r", vmin=0, vmax=1)
axes[1].set_title("Tubulin")

axes[2].imshow(clahe_both, cmap="magma", vmin=0, vmax=1)
axes[2].set_title("CLAHE(max(actin, tubulin))")

for ax in axes:
    ax.axis("off")

plt.suptitle(
    f"{row['compound_norm']} / {row['Image_Metadata_Plate_DAPI']} / {row['Image_Metadata_Well_DAPI']}",
    fontsize=12,
)

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "clahe_merged_actin_tubulin.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

In [ ]:
TUNED_CELLPOSE_PARAMS = dict(
    diameter=None,
    cellprob_threshold=0.0,
    flow_threshold=0.8,
    channels=[1, 3],
)

CELLPOSE_VARIANTS = {
    "baseline": {
        "label": "Baseline",
        "params": BASELINE_CELLPOSE_PARAMS,
    },
    "clahe": {
        "label": "CLAHE",
        "params": BASELINE_CELLPOSE_PARAMS,
    },
    "merged": {
        "label": "Actin + Tubulin",
        "params": BASELINE_CELLPOSE_PARAMS,
    },
    "tuned": {
        "label": "Tuned Cellpose",
        "params": TUNED_CELLPOSE_PARAMS,
    },
    "all_enhanced": {
        "label": "CLAHE + actin/tubulin + tuned",
        "params": TUNED_CELLPOSE_PARAMS,
    },
}

In [ ]:
def prepare_cellpose_input_variant(field: dict, variant: str) -> np.ndarray:
    """Builds (H, W, 3): R=cytoplasm signal, G=0, B=DAPI"""
    if variant in ("baseline", "tuned"):
        cyto = field["actin"]
    elif variant == "clahe":
        cyto = clahe(field["actin"])
    elif variant == "merged":
        cyto = merged_channels(field)
    elif variant == "all_enhanced":
        cyto = clache_merged_channels(field)

    return np.stack(
        [
            cyto,
            np.zeros_like(field["dapi"]),
            field["dapi"],
        ],
        axis=-1,
    ).astype(np.float32)


def segment_cellpose_variant(field: dict, variant: str) -> np.ndarray:
    """Runs Cellpose in modes"""
    img = prepare_cellpose_input_variant(field, variant)
    out = cp_model.eval(img, **CELLPOSE_VARIANTS[variant]["params"])
    return out[0].astype(np.int32)

# Run enhanced modes

In [ ]:
variants = ["baseline", "clahe", "merged", "tuned", "all_enhanced"]

for variant in variants:
    masks_by_idx = {}

    for idx, row in tqdm(plates_meta.iterrows(), total=len(plates_meta), desc=f"Cellpose {variant}"):
        cache_path = cellpose_variant_mask_path(row, variant)

        field = load_field(row)
        masks = segment_cellpose_variant(field, variant)
        np.save(cache_path, masks)
        masks_by_idx[idx] = masks

    if variant == "baseline":
        cellpose_masks = masks_by_idx
        col = "n_cells_cellpose"
    else:
        col = f"n_cells_{variant}"

    plates_meta[col] = plates_meta.index.map(
        pd.Series({idx: int(m.max()) for idx, m in masks_by_idx.items()})
    )

In [ ]:
# Change in segmented cells along variants
count_cols = {
    "CLAHE": "n_cells_clahe",
    "Actin + tubulin": "n_cells_merged",
    "Hyperparameters": "n_cells_tuned",
    "All 3": "n_cells_all_enhanced",
}

growth = (
    plates_meta
    .groupby("compound_norm")
    .apply(
        lambda g: pd.Series({
            name: 100 * (g[col].mean() - g["n_cells_cellpose"].mean()) / g["n_cells_cellpose"].mean()
            for name, col in count_cols.items()
        })
    )
    .reindex(COMPOUND_ORDER)
)

fig, ax = plt.subplots(figsize=(12, 5))

growth.plot(kind="bar", ax=ax)

ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("Cell count relative to baseline, %")
ax.set_xlabel("")
ax.grid(True, axis="y", alpha=0.3)
ax.legend(title="Mode")

for container in ax.containers:
    ax.bar_label(
        container,
        labels=[f"{v:.1f}%" for v in container.datavalues],
        fontsize=8,
        padding=2,
    )

plt.xticks(rotation=45, ha="right")
plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "cellpose_enhanced_modes_cell_count.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

In [ ]:
# visual check of modes

variants_to_show = variants

variant_titles = {
    "baseline": "Baseline",
    "clahe": "CLAHE",
    "merged": "Actin + tubulin",
    "tuned": "Hyperparameters",
    "all_enhanced": "All 3",
}

fig, axes = plt.subplots(
    len(variants_to_show),
    len(demo_rows),
    figsize=(4 * len(demo_rows), 3.2 * len(variants_to_show)),
    squeeze=False,
)

for i, variant in enumerate(variants_to_show):
    variant_label = variant_titles[variant]

    for j, row in enumerate(demo_rows):
        field = load_field(row)
        composite_img = make_rgb(field)

        if variant == "baseline":
            masks = cellpose_masks[row.name]
        else:
            masks = np.load(cellpose_variant_mask_path(row, variant))

        overlay = overlay_outlines_simple(composite_img, masks, thickness=1)

        axes[i, j].imshow(overlay)
        axes[i, j].set_title(
            f"{field['compound']}\n{variant_label}\n{masks.max()} cells",
            fontsize=8,
        )
        axes[i, j].axis("off")

plt.tight_layout()
fig.savefig(
    FIGURES_DIR / "cellpose_clahe_and_others.pdf",
    bbox_inches="tight",
    dpi=300,
)
plt.show()

# Statistical estimation

In [ ]:
variant = "tuned"
masks_by_idx = {}

for idx, row in tqdm(plates_meta.iterrows(), total=len(plates_meta), desc="Cellpose tuned"):
    cache_path = cellpose_variant_mask_path(row, variant)

    if cache_path.exists():
        masks = np.load(cache_path)
    else:
        field = load_field(row)
        masks = segment_cellpose_variant(field, variant)
        np.save(cache_path, masks)

    masks_by_idx[idx] = masks

cellpose_masks = masks_by_idx

plates_meta["n_cells_tuned"] = plates_meta.index.map(
    pd.Series({idx: int(m.max()) for idx, m in masks_by_idx.items()})
)

In [ ]:
analysis_meta = plates_meta[
    plates_meta.apply(
        lambda row: np.isclose(
            row["Image_Metadata_Concentration"],
            MIDDLE_CONCENTRATIONS[row["compound_norm"]],
        ),
        axis=1,
    )
].copy()

print(f"Analysis subset: {len(analysis_meta)} fields")

In [ ]:
display(
    analysis_meta
    .groupby(["compound_norm", "Image_Metadata_Concentration"])
    .agg(
        n_fields=("n_cells_tuned", "count"),
        mean_cells=("n_cells_tuned", "mean"),
        std_cells=("n_cells_tuned", "std"),
        total_cells=("n_cells_tuned", "sum"),
    )
    .round(1)
    .reset_index()
    .sort_values(["compound_norm", "Image_Metadata_Concentration"])
)

In [ ]:
analysis_meta[
    [
        "compound_norm",
        "Image_Metadata_Concentration",
        "Image_Metadata_Plate_DAPI",
        "Image_Metadata_Well_DAPI",
        "n_cells_tuned",
    ]
].sort_values(["compound_norm", "n_cells_tuned"])

Let us look at the graph to see how strongly the number of segmented cells varies between compounds, fields, and plates.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

sns.boxplot(
    data=analysis_meta,
    x="compound_norm",
    y="n_cells_tuned",
    order=COMPOUND_ORDER,
    color="lightgray",
    ax=ax,
)

sns.stripplot(
    data=analysis_meta,
    x="compound_norm",
    y="n_cells_tuned",
    hue="Image_Metadata_Plate_DAPI",
    order=COMPOUND_ORDER,
    size=6,
    jitter=True,
    ax=ax,
)

ax.tick_params(axis="x", rotation=45)
ax.set_ylabel("Cellpose objects per field")
ax.set_xlabel("")
ax.legend(title="Plate", bbox_to_anchor=(1.02, 1), loc="upper left")

fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "cell_counts_per_field_by_compound_and_plate.pdf",
    bbox_inches="tight",
    dpi=300,
)
plt.show()


We deliberately included DMSO from each plate in order to compare different plates through the control. For **DMSO**, there are approximately 235-345 objects per field, with a median of around 280. This is a high and relatively stable control.

**Lovastatin** and **AZ258** have far fewer objects. If this is a real effect, it can be explained by poorer cell survival under the action of the drugs. As we will see later from the mask overlays, there are indeed fewer cells. This may be a biological effect. **Etoposide** also has around 160-200 objects, with a fairly compact distribution. This looks like a moderate decrease in the number of objects relative to DMSO.

The number of cells for **AZ-U** is concerning. There is an outlier field with an abnormally high number of cells. On the one hand, after AZ-U treatment the cells do become smaller and form extensive clusters; on the other hand, this field still contains an abnormally larger number of cells than the DMSO control and far more than the other AZ-U fields, which are very close to each other. As we can see from the image with masks, there really are many cells, about twice as many as usual. Several cell clusters are very close to each other in this field.

In [ ]:
outliers = analysis_meta[
    analysis_meta["compound_norm"] == "Lovastatin"
].sort_values("n_cells_tuned")

fig, axes = plt.subplots(
    len(outliers),
    1,
    figsize=(5, 3.5 * len(outliers)),
)

axes = np.atleast_1d(axes)

for i, (idx, row) in enumerate(outliers.iterrows()):
    field = load_field(row)
    composite_img = make_rgb(field)

    masks = cellpose_masks[idx]
    overlay = overlay_outlines_simple(composite_img, masks, thickness=1)

    title = (
        f"{row['Image_Metadata_Plate_DAPI']} / "
        f"{row['Image_Metadata_Well_DAPI']} / "
        f"{row['n_cells_tuned']} cells"
    )

    axes[i].imshow(overlay)
    axes[i].set_title(title, fontsize=9)
    axes[i].axis("off")

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "outliers_lovastatin.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

In [ ]:
outliers = analysis_meta[
    analysis_meta["compound_norm"] == "AZ-U"
].sort_values("n_cells_tuned")

fig, axes = plt.subplots(
    len(outliers),
    1,
    figsize=(5, 3.5 * len(outliers)),
)

axes = np.atleast_1d(axes)

for i, (idx, row) in enumerate(outliers.iterrows()):
    field = load_field(row)
    composite_img = make_rgb(field)

    masks = cellpose_masks[idx]
    overlay = overlay_outlines_simple(composite_img, masks, thickness=1)

    title = (
        f"{row['Image_Metadata_Plate_DAPI']} / "
        f"{row['Image_Metadata_Well_DAPI']} / "
        f"{row['n_cells_tuned']} cells"
    )

    axes[i].imshow(overlay)
    axes[i].set_title(title, fontsize=9)
    axes[i].axis("off")

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "outliers_AZ-U.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

In [ ]:
analysis_meta.sort_values("n_cells_tuned").head(5)

In [ ]:
analysis_meta.sort_values("n_cells_tuned").tail(5)

# Morphometry collection


`skimage.measure.regionprops` computes dozens of geometric and intensity-based characteristics for each cell in the mask. We select the nine most interpretable ones for our task:

| Feature | What it means for a biologist |
|---|---|
| `area` | cell area in pixels |
| `perimeter` | length of the cell boundary |
| `eccentricity` | how elongated the cell is: 0 = circle, 1 = “thread” |
| `solidity` | area / convex hull area — a measure of how “irregular” the contour is |
| `extent` | area / bounding box area |
| `equivalent_diameter` | diameter of a circle with the same area |
| `major_axis_length` | length of the major axis of the ellipse fitted to the cell |
| `minor_axis_length` | length of the minor axis |
| `mean_intensity` (actin and tubulin) | average channel brightness inside the cell |

The ratio `major_axis_length / minor_axis_length` is called the **aspect ratio** — a simple measure of elongation that is more understandable than eccentricity.

`clear_border` from skimage removes labeled objects that touch the image border.

**Artifact filtering.** Segmentation always produces junk: single pixels from noise, or, conversely, a merged “monster” made up of several cells. We remove regions with `area < 200` and `area > 0.5 * height * width`.

In [ ]:
# List of properties calculated for each label (each cell) in the mask
PROPERTIES = (
    "label",
    "area",
    "perimeter",
    "eccentricity",
    "solidity",
    "extent",
    "equivalent_diameter",
    "major_axis_length",
    "minor_axis_length",
    "mean_intensity",
)

def extract_morphometry(field: dict, masks: np.ndarray, field_id: int) -> pd.DataFrame:
    """Calculate regionprops features after removing cells touching image border"""

    # Remove objects that touch the image border
    masks_inside = segmentation.clear_border(masks)

    intensity = np.stack(
        [field["actin"], field["tubulin"], field["dapi"]],
        axis=-1,
    )

    # Calculate properties for each cell
    # masks_inside defines cell regions
    # intensity_image for intensity-based properties
    table = measure.regionprops_table(
        masks_inside,
        intensity_image=intensity,
        properties=PROPERTIES,
    )

    # each row corresponds to one cell
    # each column corresponds to one measured feature
    df = pd.DataFrame(table)

    # Rename mean intensity columns
    df = df.rename(columns={
        "mean_intensity-0": "mean_actin",
        "mean_intensity-1": "mean_tubulin",
        "mean_intensity-2": "mean_dapi",
    })

    # add field metadata to group cells
    df["field_id"] = field_id
    df["compound"] = field["compound"]
    df["concentration"] = field["concentration"]
    df["plate"] = field["plate"]
    df["well"] = field["well"]

    # Count objects before clear_border
    # np.unique(masks) returns all label values, 0 for background
    # subtract 1 so background is not counted as a cell
    n_before = len(np.unique(masks)) - 1
    # Count objects after removing border cells
    n_after = len(np.unique(masks_inside)) - 1
    df["n_border_dropped"] = n_before - n_after

    df["aspect_ratio"] = (
        df["major_axis_length"] / df["minor_axis_length"].replace(0, np.nan)
    )

    return df

In [ ]:
# Masks from cache/masks/*__cellpose_tuned.npy
all_morph: list[pd.DataFrame] = []

for idx, row in tqdm(
    analysis_meta.iterrows(),
    total=len(analysis_meta),
    desc="regionprops tuned",
):
    f = load_field(row)

    mask_path = cellpose_variant_mask_path(row, "tuned")
    if not mask_path.exists():
        raise FileNotFoundError(f"Missing tuned mask: {mask_path}")

    masks = np.load(mask_path)

    df = extract_morphometry(f, masks, field_id=idx)
    all_morph.append(df)

morph = pd.concat(all_morph, ignore_index=True)

n_border_total = int(
    morph.groupby("field_id")["n_border_dropped"].first().sum()
)

print(f"Found {len(morph)} cells after clear_border; {n_border_total} border objects removed.")

In [ ]:
morph_raw = morph.copy()
morph_raw.groupby("compound")["area"].quantile(
    [0.01, 0.05, 0.5, 0.95, 0.99]
).unstack().reindex(COMPOUND_ORDER)

We remove objects that are too large, in cases where several cells have merged into clusters, and objects that are too small, which are likely artefacts or noise. To estimate the thresholds, we use the expected diameters returned by Cellpose and approximate the areas using the formula:

```text
area ≈ π * (diameter / 2) ** 2
```

```text
diameter 43.0  -> area ≈ 1,452 px
diameter 56.7  -> area ≈ 2,526 px
diameter 64.2  -> area ≈ 3,238 px
diameter 80.7  -> area ≈ 5,115 px
diameter 96.3  -> area ≈ 7,284 px
```

Thus, the expected size of one normal cell is approximately **1,500-7,300 px**, if estimated as a circle. We remove objects smaller than 700 px and larger than 15,000 px. The lower threshold is approximately half of the smallest expected Cellpose-derived cell area, while the upper threshold is about twice the largest expected area. This keeps a conservative margin.

In [ ]:
# filtering very small and very large objects
example_idx = analysis_meta.index[0]
example_row = analysis_meta.loc[example_idx]
H, W = np.load(cellpose_variant_mask_path(example_row, "tuned")).shape

MIN_AREA = 300
MAX_AREA = 25000

n_before = len(morph)

mask_keep = morph["area"].between(MIN_AREA, MAX_AREA)
morph = morph.loc[mask_keep].reset_index(drop=True)

print(
    f"After filtering (area in [{MIN_AREA}, {int(MAX_AREA)}]): "
    f"{len(morph)} cells kept ({n_before - len(morph)} removed)."
)

print()
print("Cell count by compound:")
print(
    morph["compound"]
    .value_counts()
    .reindex(COMPOUND_ORDER)
    .fillna(0)
    .astype(int)
)

In [ ]:
# save morphometry
morph.to_parquet(CACHE_DIR / "morphometry.parquet", index=False)

print(f"Saved: {CACHE_DIR / 'morphometry.parquet'}")
print()

morph.head(10)

Before plotting graphs for thousands of cells, we annotate each cell on the colored mask with its area and eccentricity. If the values are reasonable — the area visually matches the cell size, and elongated cells have higher eccentricity than round cells — then the morphometry can be trusted.

In [ ]:
def overlay_props(field: dict, masks: np.ndarray, ax) -> None:
    masks_inside = segmentation.clear_border(masks)
    rgb = label2rgb(masks_inside, bg_label=0, alpha=0.45)
    composite = make_rgb(field)
    blended = 0.55 * composite + 0.45 * rgb
    ax.imshow(np.clip(blended, 0, 1))

    for prop in measure.regionprops(masks_inside):
        if prop.area < 1000:
            continue

        cy, cx = prop.centroid

        ax.text(
            cx,
            cy,
            f"A={int(prop.area)}\ne={prop.eccentricity:.2f}",
            ha="center",
            va="center",
            fontsize=7,
            color="white",
            bbox=dict(
                boxstyle="round,pad=0.15",
                fc="black",
                ec="none",
                alpha=0.55,
            ),
        )

    ax.axis("off")


fig, axes = plt.subplots(
    1,
    len(demo_rows),
    figsize=(7 * len(demo_rows), 7),
    squeeze=False,
)

for ax, row in zip(axes.ravel(), demo_rows):
    idx = row.name

    field = load_field(row)
    masks = np.load(cellpose_variant_mask_path(row, "tuned"))

    overlay_props(field, masks, ax)

    ax.set_title(
        f"{row['compound_norm']}"
    )

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "morphometry_demo.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

Let us compare several cells that appear to have a similar shape by eye — their `area` and `eccentricity` should be close to each other. A strongly elongated cell should have `eccentricity > 0.8`, while an almost round cell should have `< 0.4`. If a round cell has an `eccentricity` of 0.9, this is a sign that something is wrong with the mask, for example two cells may have merged into one elongated object.

# EDA: how compounds change cells

We now have a cell × features table. This allows us to address the main question of the project: **how cell morphology differs under different drug treatments.**

Strategy:

1. **Feature distributions** — violin plots for features.
2. **Summary heatmap** — z-normalized mean feature values by compound.
3. **Dimensionality reduction (PCA)** — project cells into a lower-dimensional space and check whether different treatments form separate clusters.
4. **Pairplots** — whether treatments separate not only by individual features, but also by combinations of size, shape, and cytoskeletal intensity.

In [ ]:
# Feature distribution along drugs

FEATURES = [
    "area",
    "perimeter",
    "eccentricity",
    "solidity",
    "extent",
    "equivalent_diameter",
    "major_axis_length",
    "minor_axis_length",
    "aspect_ratio",
    "mean_actin",
    "mean_tubulin",
]

COMPOUND_PRESENT = [row["compound_norm"] for row in demo_rows]

n_features = len(FEATURES)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(16, 4.2 * n_rows),
    squeeze=False,
)

axes = axes.ravel()
palette = dict(zip(COMPOUND_PRESENT, sns.color_palette("Set2", len(COMPOUND_PRESENT))))

for i, feature in enumerate(FEATURES):
    ax = axes[i]

    sns.violinplot(
        data=morph,
        x="compound",
        y=feature,
        hue="compound",
        order=COMPOUND_PRESENT,
        hue_order=COMPOUND_PRESENT,
        palette=palette,
        legend=False,
        inner="quartile",
        cut=0,
        linewidth=1.0,
        ax=ax,
    )

    ax.set_title(feature)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, axis="y", alpha=0.3)

for j in range(n_features, len(axes)):
    axes[j].axis("off")

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "morphometry_violin.pdf",
    bbox_inches="tight",
    dpi=300,
)
plt.show()

In [ ]:
# median feature values by compound
summary_medians = (
    morph
    .replace([np.inf, -np.inf], np.nan)
    .groupby("compound")[FEATURES]
    .median()
    .reindex(COMPOUND_PRESENT)
    .round(2)
)

summary_medians

In [ ]:
# Difference in median feature values relative to DMSO
summary_vs_dmso = summary_medians.subtract(
    summary_medians.loc["DMSO"],
    axis="columns",
).round(2)

summary_vs_dmso

In [ ]:
# difference in median feature values relative to DMSO

summary_vs_dmso_plot = summary_vs_dmso.drop(index="DMSO")

fig, axes = plt.subplots(
    3,
    4,
    figsize=(16, 10),
    squeeze=False,
)

axes = axes.ravel()

for i, feature in enumerate(FEATURES):
    ax = axes[i]

    summary_vs_dmso_plot[feature].plot(
        kind="bar",
        ax=ax,
        color="steelblue",
        edgecolor="black",
        linewidth=0.5,
    )

    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(feature)
    ax.set_xlabel("")
    ax.set_ylabel("Median difference")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, axis="y", alpha=0.25)

for j in range(len(FEATURES), len(axes)):
    axes[j].axis("off")

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "median_difference_vs_dmso.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

In [ ]:
# Phenotypic fingerprint: mean features by compound (z-score)
# compact phenotypic fingerprint

group_mean = morph.groupby("compound")[FEATURES].mean()

zscored = (group_mean - morph[FEATURES].mean()) / morph[FEATURES].std()
zscored = zscored.loc[COMPOUND_PRESENT]

fig, ax = plt.subplots(figsize=(10, 4.5))

sns.heatmap(
    zscored,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    cbar_kws={"label": "Z-score relative to the global mean"},
    ax=ax,
)

ax.set_xlabel("Feature")
ax.set_ylabel("Compound")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "morphometry_heatmap.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

In [ ]:
# 3D PCA
X = morph[FEATURES].to_numpy()
mask_valid = np.isfinite(X).all(axis=1)

X = X[mask_valid]
morph_pca = morph.loc[mask_valid].copy()

groups = morph_pca["compound"].to_numpy()

scaler = StandardScaler()
X_std = scaler.fit_transform(X)

pca = PCA(n_components=3, random_state=SEED)
X_pca = pca.fit_transform(X_std)

pca_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2", "PC3"],
    index=morph_pca.index,
)

pca_df["compound"] = morph_pca["compound"].values

hover_cols = [
    col for col in ["field_id", "plate", "well", "concentration", "label"]
    if col in morph_pca.columns
]

for col in hover_cols:
    pca_df[col] = morph_pca[col].values

total_var = pca.explained_variance_ratio_.sum() * 100

fig = px.scatter_3d(
    pca_df,
    x="PC1",
    y="PC2",
    z="PC3",
    color="compound",
    category_orders={"compound": COMPOUND_PRESENT},
    hover_data=hover_cols,
    opacity=0.65,
    title=f"Total explained variance: {total_var:.1f}%",
    labels={
        "PC1": f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)",
        "PC2": f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)",
        "PC3": f"PC3 ({pca.explained_variance_ratio_[2] * 100:.1f}%)",
        "compound": "Compound",
    },
)

fig.update_traces(marker=dict(size=4))
fig.update_layout(
    width=1050,
    height=750,
    legend=dict(
        title="Compound",
        x=1.02,
        y=1.0,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="lightgray",
        borderwidth=1,
    ),
    margin=dict(l=0, r=180, t=80, b=0),
)

fig.write_html(
    FIGURES_DIR / "pca_3d.html",
    include_plotlyjs="cdn",
)

fig.show()

In [ ]:
# PC1 vs PC2

pca_data = morph.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES).copy()

X = pca_data[FEATURES].to_numpy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2"],
    index=pca_data.index,
)

pca_df["compound"] = pca_data["compound"].values

fig, ax = plt.subplots(figsize=(8, 6))

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="compound",
    hue_order=COMPOUND_PRESENT,
    s=18,
    alpha=1,
    linewidth=0,
    ax=ax,
)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)")
ax.grid(True, alpha=0.25)

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "pc1_pc2.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

In [ ]:
# PC1 vs PC3

pca_data = morph.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES).copy()

X = pca_data[FEATURES].to_numpy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=3, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2", "PC3"],
    index=pca_data.index,
)

pca_df["compound"] = pca_data["compound"].values

fig, ax = plt.subplots(figsize=(8, 6))

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC3",
    hue="compound",
    hue_order=COMPOUND_PRESENT,
    s=18,
    alpha=1,
    linewidth=0,
    ax=ax,
)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)")
ax.set_ylabel(f"PC3 ({pca.explained_variance_ratio_[2] * 100:.1f}%)")
ax.grid(True, alpha=0.25)

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "pc1_pc3.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

In [ ]:
# Feature loadings on the principal components

loadings = pd.DataFrame(
    pca.components_.T,
    index=FEATURES,
    columns=["PC1", "PC2", "PC3"],
)
display(loadings.round(2))

In [ ]:
# Features with largest contribution
display(
    pd.concat(
        {
            pc: loadings[pc].abs().sort_values(ascending=False).head(5)
            for pc in ["PC1", "PC2", "PC3"]
        },
        axis=1,
    ).round(2)
)

In [ ]:
# Features with largest contribution
top_loadings = pd.DataFrame({
    pc: [
        f"{feature}: {value:.3f}"
        for feature, value in loadings[pc]
            .abs()
            .sort_values(ascending=False)
            .head(5)
            .items()
    ]
    for pc in ["PC1", "PC2", "PC3"]
})

display(top_loadings)

In [ ]:
# Pairwise plots of some features

PAIR_FEATURES = [
    ("equivalent_diameter", "aspect_ratio"),
    ("area", "solidity"),
    ("equivalent_diameter", "mean_tubulin"),
]

pairplot_data = (
    morph
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=[f for pair in PAIR_FEATURES for f in pair] + ["compound"])
    .copy()
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4), squeeze=False)
axes = axes.ravel()

for ax, (x_feature, y_feature) in zip(axes, PAIR_FEATURES):
    sns.scatterplot(
        data=pairplot_data,
        x=x_feature,
        y=y_feature,
        hue="compound",
        s=12,
        alpha=1,
        linewidth=0,
        ax=ax,
    )

    ax.set_xlabel(x_feature)
    ax.set_ylabel(y_feature)
    ax.grid(True, alpha=0.25)

    if ax is axes[0]:
        ax.legend(
            title="Compound",
            loc="upper right",
            frameon=True,
        )
    else:
        ax.get_legend().remove()

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "morphometry_pca_informative_pairs.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

# Classifiers

We will build simple classifiers on the raw features to try to distinguish, based on morphology, the classes of compounds used to treat the cells.

First, it is important to split the data into train and test sets to avoid leakage. We split by `field_id`, stratified by `compound`. This means that cells from the same field of view will not appear in both train and test at the same time. We have 4 fields per compound, so 3 fields will go into the training set and 1 field into the test set for each compound. If we had more data, we could also stratify by well, since wells may not be independent.

We will train four classifiers: Logistic Regression as a simple linear baseline, Random Forest as a nonlinear baseline, SVM as a margin-based nonlinear classifier, and Decision Tree as an interpretable model.

Separately, we will also train a drug-only SVM without DMSO to test how well the model distinguishes only active treatments, without the control class.

Main metrics:

`precision` — among the cells that the model predicted as a given compound, the fraction that truly belonged to that compound.

`recall` — among all true cells of a given compound, the fraction that the model correctly found.

`f1-score` — a balance between precision and recall.

`support` — the number of true test cells in this class.

`accuracy` — the overall fraction of correctly classified test cells across all classes.

`macro avg` — the simple average of the metric across classes, where each class has equal weight regardless of how many cells it contains.

`weighted avg` — the average of the metric across classes weighted by `support`, so classes with more test cells contribute more.

In [ ]:
# prepare split by field_id
morph_ml = pd.read_parquet(CACHE_DIR / "morphometry.parquet").copy()

field_units = (
    morph_ml
    .groupby("field_id")
    .agg(
        compound=("compound", "first"),
        n_compounds=("compound", "nunique"),
        plate=("plate", "first"),
        well=("well", "first"),
        n_cells=("label", "count"),
    )
    .reset_index()
)

display(
    field_units
    .groupby("compound")
    .agg(
        n_fields=("field_id", "count"),
        n_cells=("n_cells", "sum"),
    )
    .reindex(COMPOUND_PRESENT)
)

In [ ]:
display(
    field_units
    .groupby(["compound", "plate", "well"])
    .agg(
        n_fields=("field_id", "count"),
        n_cells=("n_cells", "sum"),
    )
    .reset_index()
    .sort_values(["compound", "plate", "well"])
)

In [ ]:
train_fields_df, test_fields_df = train_test_split(
    field_units,
    test_size=0.25,
    random_state=SEED,
    stratify=field_units["compound"],
)

train_fields = set(train_fields_df["field_id"])
test_fields = set(test_fields_df["field_id"])

train_mask = morph_ml["field_id"].isin(train_fields)
test_mask = morph_ml["field_id"].isin(test_fields)

X_train = morph_ml.loc[train_mask, FEATURES]
X_test = morph_ml.loc[test_mask, FEATURES]
y_train = morph_ml.loc[train_mask, "compound"]
y_test = morph_ml.loc[test_mask, "compound"]

valid_train = np.isfinite(X_train.to_numpy()).all(axis=1)
valid_test = np.isfinite(X_test.to_numpy()).all(axis=1)

X_train = X_train.loc[valid_train]
y_train = y_train.loc[valid_train]

X_test = X_test.loc[valid_test]
y_test = y_test.loc[valid_test]

print("Train fields:", len(train_fields), "       Test fields:", len(test_fields))
print("Train cells:", len(X_train), "      Test cells:", len(X_test))
print()

In [ ]:
split_summary = pd.DataFrame({
    "compound": COMPOUND_PRESENT,
    "train_fields": train_fields_df["compound"].value_counts().reindex(COMPOUND_PRESENT, fill_value=0).values,
    "test_fields": test_fields_df["compound"].value_counts().reindex(COMPOUND_PRESENT, fill_value=0).values,
    "train_cells": y_train.value_counts().reindex(COMPOUND_PRESENT, fill_value=0).values,
    "test_cells": y_test.value_counts().reindex(COMPOUND_PRESENT, fill_value=0).values,
})

display(split_summary)

In [ ]:
print("Train wells:")
display(
    train_fields_df[["compound", "plate", "well"]]
    .drop_duplicates()
    .sort_values(["compound", "plate", "well"])
)

print("Test wells:")
display(
    test_fields_df[["compound", "plate", "well"]]
    .drop_duplicates()
    .sort_values(["compound", "plate", "well"])
)

It is important to note that the classifiers here are used as a technical demonstration. They may be biased because there is leakage at the well level. To make the analysis strictly correct, the dataset would need to be expanded so that each compound is represented by several independent wells and/or plates, followed by a train/test split at the plate + well level. This would substantially increase the amount of data to download and process, so in this project we focus on demonstrating the feasibility of this analysis design, while further improvement would be required.

We first build a Logistic Regression classifier. We use `StandardScaler()`, which transforms each feature as `x_scaled = (x - mean) / std`. In other words, scaling makes each feature have mean 0 and standard deviation 1. This is needed to put features with different numerical ranges on a comparable scale. `class_weight="balanced"` compensates for class imbalance, so mistakes on rarer classes are penalized more strongly. By default, `penalty="l2"` is used: it penalizes overly large model coefficients and helps reduce overfitting. Also by default, the model uses standard moderate regularization with `C=1.0`: smaller `C` means stronger regularization, while larger `C` means weaker regularization. The parameters were selected by grid search, so they can be considered suitable for our task.

In [ ]:
# Logistic Regression

logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=SEED,
    )),
])

logreg.fit(X_train, y_train)

y_pred = logreg.predict(X_test)

print(classification_report(
    y_test,
    y_pred,
    labels=COMPOUND_PRESENT,
    zero_division=0,
))

cm = confusion_matrix(y_test, y_pred, labels=COMPOUND_PRESENT)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=COMPOUND_PRESENT,
).plot(ax=ax, cmap="Blues", colorbar=False)

ax.set_title("Logistic Regression")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "log_reg_conf_matrix.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

Random Forest classifier is an ensemble of many decision trees. Each tree tries to classify a cell based on its morphometric features, and the final prediction is made by voting across the trees. This classifier can capture nonlinear relationships and combinations of features. For example, if `area` is large, `mean_tubulin` is low, and `solidity` is high, the cell may be more likely to belong to `AZ258`.

`n_estimators=300` defines the number of trees in the forest. This value was selected experimentally. The same applies to `max_depth=8`. Tree depth means the number of consecutive split decisions from the root of the tree to a final leaf. In our case, values from 4 to 8 were optimal: with 5 classes, depths below 4 may be too shallow to separate the data well, while depths above 8 started to increase overfitting.

Random Forest uses bootstrap samples. A bootstrap sample is a training subset created by sampling from the original training data with replacement. This means that each tree is trained on a slightly different version of the training set: some cells may appear more than once, while others may be left out. This makes the trees less identical and improves the stability of the ensemble.

`max_features="sqrt"` means that at each split, a tree does not consider all features, but only a random subset of size `sqrt(number_of_features)`. In our case, `sqrt(11) ≈ 3`. This also makes the trees less similar to each other. The ensemble works better because individual trees make different errors.

A leaf is the final node of a tree where the prediction is made. The smaller `min_samples_leaf` is, the narrower and more specific the rules can become, which increases the risk of overfitting. The larger this parameter is, the less detailed the tree becomes, but the better it may generalize. Thus, `min_samples_leaf=10` prevents the trees from creating rules based on individual cells.

In [ ]:
# Random Forest

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
    max_depth=8,
    max_features="sqrt",
    min_samples_leaf=10,
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print(classification_report(
    y_test,
    y_pred_rf,
    labels=COMPOUND_PRESENT,
    zero_division=0,
))

cm = confusion_matrix(y_test, y_pred_rf, labels=COMPOUND_PRESENT)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=COMPOUND_PRESENT,
).plot(ax=ax, cmap="Greens", colorbar=False)

ax.set_title("Random Forest")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "rf_conf_matrix.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

We will try a Support Vector Machine with an RBF kernel, which allows the model to build nonlinear boundaries between classes. The idea is to maximize the margin between the decision boundary and the closest training points, called support vectors. Here we also scale the features, because SVM relies on distances between points. The RBF kernel allows the model to separate classes using complex curved boundaries.

The regularization parameter `C=1.0` controls the tradeoff between a simpler decision boundary and fewer errors on the training set. A smaller `C` means stronger regularization, producing a simpler and more stable boundary. A larger `C` means weaker regularization: the model tries harder to classify the training points correctly, which increases the risk of overfitting.

`gamma` controls how local the influence of individual training points is. A small `gamma` gives a smoother, broader decision boundary; a large `gamma` gives a more complex boundary that adapts more strongly to individual points. `"scale"` means that sklearn calculates `gamma` automatically using a formula based on the number of features and the variance of the data.

In [ ]:
# Support Vector Machine

svm = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced",
        random_state=SEED,
    )),
])

svm.fit(X_train, y_train)

y_pred_svm = svm.predict(X_test)

print(classification_report(
    y_test,
    y_pred_svm,
    labels=COMPOUND_PRESENT,
    zero_division=0,
))

cm = confusion_matrix(y_test, y_pred_svm, labels=COMPOUND_PRESENT)

fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=COMPOUND_PRESENT,
).plot(ax=ax, cmap="Purples", colorbar=False)

ax.set_title("SVM")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "svm_conf_matrix.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

Across all classifiers, the confusion matrices show that the models are not guessing randomly: most values lie on the diagonal, meaning that many cells are classified correctly. `AZ-U` is separated best.

The main problem is that not all compounds have a unique morphometric profile based on these 11 features. `DMSO` is often confused with `Etoposide` and `Lovastatin`; `Lovastatin` is noticeably confused with `AZ-U`; and the strongest pair of errors is `Etoposide` versus `AZ258`.

This pattern is repeated across all classifiers, which suggests that the limitation is more likely in the data or in the feature set rather than in the models themselves. The morphometric features provide a useful signal, but it is not sufficient to fully separate all treatment phenotypes at the single-cell level.

In [ ]:
# SVM decision regions on PC1/PC2

x_min, x_max = pca_df["PC1"].min() - 1, pca_df["PC1"].max() + 1
y_min, y_max = pca_df["PC2"].min() - 1, pca_df["PC2"].max() + 1

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 400),
    np.linspace(y_min, y_max, 400),
)

grid_pca = np.c_[xx.ravel(), yy.ravel(), np.zeros(xx.size)]

# Convert the PC1/PC2 grid back to the original feature space
grid_scaled = pca.inverse_transform(grid_pca)
grid_original = scaler.inverse_transform(grid_scaled)

grid_df = pd.DataFrame(grid_original, columns=FEATURES)

grid_pred = svm.predict(grid_df)

class_to_id = {compound: i for i, compound in enumerate(COMPOUND_PRESENT)}
Z = np.array([class_to_id[c] for c in grid_pred]).reshape(xx.shape)

palette = sns.color_palette("Set2", len(COMPOUND_PRESENT))
cmap = ListedColormap(palette)

fig, ax = plt.subplots(figsize=(8, 6))

ax.contourf(
    xx,
    yy,
    Z,
    levels=np.arange(len(COMPOUND_PRESENT) + 1) - 0.5,
    cmap=cmap,
    alpha=0.25,
)

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="compound",
    hue_order=COMPOUND_PRESENT,
    palette=palette,
    s=18,
    alpha=1,
    linewidth=0,
    ax=ax,
)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)")
ax.grid(True, alpha=0.25)

ax.legend(
    title="Compound",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "svm_decision_regions_pc1_pc2.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

We will also build a Decision Tree to visualize the decision structure as a tree and see how the features are split. The tree uses the Gini criterion to choose the best split. A depth of 4-6 is optimal here: lower values may lead to underfitting, while higher values may lead to overfitting.

In [ ]:
# Decision Tree Classifier

tree = DecisionTreeClassifier(
    max_depth=4,
    class_weight="balanced",
    random_state=SEED,
)

tree.fit(X_train, y_train)

y_pred_tree = tree.predict(X_test)

print(classification_report(
    y_test,
    y_pred_tree,
    labels=COMPOUND_PRESENT,
    zero_division=0,
))

cm = confusion_matrix(y_test, y_pred_tree, labels=COMPOUND_PRESENT)

fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=COMPOUND_PRESENT,
).plot(ax=ax, cmap="Oranges", colorbar=False)

ax.set_title("Decision Tree")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "dt_conf_matrix.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

In [ ]:
# DT with dtreeviz

class_to_id = {
    compound: i
    for i, compound in enumerate(COMPOUND_PRESENT)
}

y_train_tree_viz = y_train.map(class_to_id).astype(int)

tree_viz = DecisionTreeClassifier(
    max_depth=4,
    class_weight="balanced",
    random_state=SEED,
)

tree_viz.fit(X_train, y_train_tree_viz)

viz_model = dtreeviz.model(
    tree_viz,
    X_train=X_train,
    y_train=y_train_tree_viz,
    feature_names=FEATURES,
    target_name="compound",
    class_names=COMPOUND_PRESENT,
)

viz = viz_model.view(fontname="DejaVu Sans")

svg_path = FIGURES_DIR / "dt_tree_dtreeviz.svg"
viz.save(str(svg_path))

display(SVG(filename=str(svg_path)))

In [ ]:
# Drug-only classifier

ACTIVE_COMPOUNDS = [
    c for c in COMPOUND_PRESENT
    if c != "DMSO"
]

morph_drug = morph_ml[morph_ml["compound"].isin(ACTIVE_COMPOUNDS)].copy()

train_fields_df_drug = train_fields_df[
    train_fields_df["compound"].isin(ACTIVE_COMPOUNDS)
].copy()

test_fields_df_drug = test_fields_df[
    test_fields_df["compound"].isin(ACTIVE_COMPOUNDS)
].copy()

train_fields_drug = set(train_fields_df_drug["field_id"])
test_fields_drug = set(test_fields_df_drug["field_id"])

train_mask_drug = morph_drug["field_id"].isin(train_fields_drug)
test_mask_drug = morph_drug["field_id"].isin(test_fields_drug)

X_train_drug = morph_drug.loc[train_mask_drug, FEATURES]
X_test_drug = morph_drug.loc[test_mask_drug, FEATURES]
y_train_drug = morph_drug.loc[train_mask_drug, "compound"]
y_test_drug = morph_drug.loc[test_mask_drug, "compound"]

valid_train_drug = np.isfinite(X_train_drug.to_numpy()).all(axis=1)
valid_test_drug = np.isfinite(X_test_drug.to_numpy()).all(axis=1)

X_train_drug = X_train_drug.loc[valid_train_drug]
y_train_drug = y_train_drug.loc[valid_train_drug]

X_test_drug = X_test_drug.loc[valid_test_drug]
y_test_drug = y_test_drug.loc[valid_test_drug]

print("Train fields:", len(train_fields_drug), "      Test fields:", len(test_fields_drug))
print("Train cells:", len(X_train_drug), "     Test cells:", len(X_test_drug))

In [ ]:
display(pd.DataFrame({
    "compound": ACTIVE_COMPOUNDS,
    "train_fields": train_fields_df_drug["compound"].value_counts().reindex(ACTIVE_COMPOUNDS, fill_value=0).values,
    "test_fields": test_fields_df_drug["compound"].value_counts().reindex(ACTIVE_COMPOUNDS, fill_value=0).values,
    "train_cells": y_train_drug.value_counts().reindex(ACTIVE_COMPOUNDS, fill_value=0).values,
    "test_cells": y_test_drug.value_counts().reindex(ACTIVE_COMPOUNDS, fill_value=0).values,
}))

In [ ]:
svm_drug = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced",
        random_state=SEED,
    )),
])

svm_drug.fit(X_train_drug, y_train_drug)

y_pred_svm_drug = svm_drug.predict(X_test_drug)

print(classification_report(
    y_test_drug,
    y_pred_svm_drug,
    labels=ACTIVE_COMPOUNDS,
    zero_division=0,
))

cm = confusion_matrix(
    y_test_drug,
    y_pred_svm_drug,
    labels=ACTIVE_COMPOUNDS,
)

fig, ax = plt.subplots(figsize=(5.5, 5))

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=ACTIVE_COMPOUNDS,
).plot(ax=ax, cmap="Purples", colorbar=False)

ax.set_title("Drug-only SVM")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "svm_drug_conf_matrix.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

In the drug-only SVM without `DMSO`, the model still separates `AZ-U` well. `Etoposide` and `AZ258` remain a closely related pair. The main new observation is that `Lovastatin` is strongly confused with `AZ-U`. Thus, removing the control class does not solve the problem of similar morphometric profiles among active treatments; instead, it highlights it.

In [ ]:
# Drug-only Random Forest

rf_drug = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
    max_depth=8,
    max_features="sqrt",
    min_samples_leaf=10,
)

rf_drug.fit(X_train_drug, y_train_drug)

y_pred_rf_drug = rf_drug.predict(X_test_drug)

print(classification_report(
    y_test_drug,
    y_pred_rf_drug,
    labels=ACTIVE_COMPOUNDS,
    zero_division=0,
))

cm = confusion_matrix(
    y_test_drug,
    y_pred_rf_drug,
    labels=ACTIVE_COMPOUNDS,
)

fig, ax = plt.subplots(figsize=(5.5, 5))

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=ACTIVE_COMPOUNDS,
).plot(ax=ax, cmap="Greens", colorbar=False)

ax.set_title("Drug-only Random Forest")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "rf_drug_conf_matrix.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

# Dose response

Let us check whether cell morphology differs across concentrations. We originally downloaded the subset so that each compound, except DMSO, had three concentrations. For the previous analysis, we used the middle concentration. Now we will use all three concentrations and plot the number of segmented cells and morphological features as a function of concentration.

In [ ]:
# Dose-response morphology analysis

cell_count_by_field = {}
all_morph_by_concentration = []

for idx, row in tqdm(
    plates_meta.iterrows(),
    total=len(plates_meta),
    desc="regionprops tuned by concentration",
):
    field = load_field(row)
    mask_path = cellpose_variant_mask_path(row, "tuned")
    masks = np.load(mask_path)
    cell_count_by_field[idx] = int(masks.max())
    df = extract_morphometry(field, masks, field_id=idx)
    all_morph_by_concentration.append(df)


plates_meta["n_cells_tuned"] = plates_meta.index.map(
    pd.Series(cell_count_by_field)
)

morph_by_concentration_raw = pd.concat(
    all_morph_by_concentration,
    ignore_index=True,
)

n_before = len(morph_by_concentration_raw)

morph_by_concentration = (
    morph_by_concentration_raw[
        morph_by_concentration_raw["area"].between(MIN_AREA, MAX_AREA)
    ]
    .reset_index(drop=True)
)

print(
    f"After filtering area in [{MIN_AREA}, {MAX_AREA}]: "
    f"{len(morph_by_concentration)} cells kept "
    f"({n_before - len(morph_by_concentration)} removed)"
)

In [ ]:
morph_by_concentration.to_parquet(
    CACHE_DIR / "morphometry_by_concentration.parquet",
    index=False,
)

display(
    morph_by_concentration
    .groupby(["compound", "concentration"], as_index=False)
    .agg(
        n_fields=("field_id", "nunique"),
        n_cells=("label", "count"),
        median_area=("area", "median"),
        median_equivalent_diameter=("equivalent_diameter", "median"),
        median_eccentricity=("eccentricity", "median"),
        median_mean_actin=("mean_actin", "median"),
        median_mean_tubulin=("mean_tubulin", "median"),
    )
    .sort_values(["compound", "concentration"])
)

In [ ]:
# Segmented cell count across concentrations

fig, axes = plt.subplots(
    1,
    len(ACTIVE_COMPOUNDS),
    figsize=(4 * len(ACTIVE_COMPOUNDS), 4),
    sharey=True,
)

axes = np.atleast_1d(axes)

for ax, compound in zip(axes, ACTIVE_COMPOUNDS):
    sub = (
        plot_meta[plot_meta["compound_norm"] == compound]
        .sort_values("Image_Metadata_Concentration")
        .copy()
    )

    concentration_order = sorted(sub["Image_Metadata_Concentration"].unique())
    concentration_labels = [f"{x:g}" for x in concentration_order]
    sub["concentration_label"] = sub["Image_Metadata_Concentration"].map(lambda x: f"{x:g}")

    sns.stripplot(
        data=sub,
        x="concentration_label",
        y="n_cells_tuned",
        order=concentration_labels,
        color="black",
        alpha=0.7,
        size=5,
        jitter=True,
        ax=ax,
    )

    medians = (
        sub
        .groupby("Image_Metadata_Concentration", as_index=False)["n_cells_tuned"]
        .median()
        .sort_values("Image_Metadata_Concentration")
    )

    ax.plot(
        range(len(medians)),
        medians["n_cells_tuned"],
        color="red",
        marker="o",
        linewidth=2,
        label="Median",
    )

    ax.set_title(compound)
    ax.set_xlabel("Concentration")
    ax.set_ylabel("Segmented cells per field")
    ax.grid(True, axis="y", alpha=0.3)

axes[0].legend()

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "dose_response_cell_counts.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

In [ ]:
field_features_by_concentration = (
    morph_by_concentration
    .groupby(["field_id", "compound", "concentration"], as_index=False)
    .agg(
        n_cells=("label", "count"),
        **{feature: (feature, "median") for feature in FEATURES},
    )
)

for feature_to_plot in FEATURES:
    dmso_reference = (
        field_features_by_concentration
        .loc[
            field_features_by_concentration["compound"] == "DMSO",
            feature_to_plot,
        ]
        .median()
    )

    fig, axes = plt.subplots(
        1,
        len(ACTIVE_COMPOUNDS),
        figsize=(4 * len(ACTIVE_COMPOUNDS), 4),
        sharey=False,
    )

    axes = np.atleast_1d(axes)

    for ax, compound in zip(axes, ACTIVE_COMPOUNDS):
        sub = (
            field_features_by_concentration[
                field_features_by_concentration["compound"] == compound
            ]
            .sort_values("concentration")
            .copy()
        )

        concentration_order = sorted(sub["concentration"].unique())
        concentration_labels = [f"{x:g}" for x in concentration_order]
        sub["concentration_label"] = sub["concentration"].map(lambda x: f"{x:g}")

        sns.stripplot(
            data=sub,
            x="concentration_label",
            y=feature_to_plot,
            order=concentration_labels,
            color="black",
            alpha=0.7,
            size=5,
            jitter=True,
            ax=ax,
        )

        medians = (
            sub
            .groupby("concentration", as_index=False)[feature_to_plot]
            .median()
            .sort_values("concentration")
        )

        ax.plot(
            range(len(medians)),
            medians[feature_to_plot],
            color="red",
            marker="o",
            linewidth=2,
            label="Median",
        )

        ax.axhline(
            dmso_reference,
            color="gray",
            linestyle="--",
            linewidth=1,
            label="DMSO median",
        )

        ax.set_title(compound)
        ax.set_xlabel("Concentration")
        ax.set_ylabel(feature_to_plot)
        ax.grid(True, axis="y", alpha=0.3)

    axes[0].legend()

    plt.suptitle(
        f"{feature_to_plot}",
        y=1.03,
        fontsize=14,
    )

    plt.tight_layout()

    fig.savefig(
        FIGURES_DIR / f"dose_response_{feature_to_plot}.pdf",
        bbox_inches="tight",
        dpi=300,
    )

    plt.show()